# Pure JAX PPO for Orbit Wars

Self-contained notebook for Kaggle GPU (e.g. 2x T4).
Runs JAX-based environment and purejaxrl-based PPO training.


## Setup Packages and Directories


In [ ]:
!pip install -q distrax gymnax
import os
os.makedirs('src/orbit_wars', exist_ok=True)


## Environment and Policy Source Code


In [ ]:
%%writefile src/orbit_wars/constants.py
"""Orbit Wars simulation constants (match official Kaggle env)."""

from __future__ import annotations

BOARD_SIZE = 100.0
CENTER = BOARD_SIZE / 2.0
SUN_RADIUS = 10.0
ROTATION_RADIUS_LIMIT = 50.0
COMET_RADIUS = 1.0
COMET_PRODUCTION = 1
PLANET_CLEARANCE = 7
MIN_PLANET_GROUPS = 5
MAX_PLANET_GROUPS = 10
MIN_STATIC_GROUPS = 3
COMET_SPAWN_STEPS = (50, 150, 250, 350, 450)
DEFAULT_SHIP_SPEED = 6.0
DEFAULT_EPISODE_STEPS = 500
MIN_LAUNCH_SHIPS = 5

# Decoding / Planning constants
SUN_PATH_MARGIN = 1.5
PATH_PLANET_MARGIN = 1.0
INTERCEPT_ITERATIONS = 5
BUCKET_COUNT = 4

# Padded simulation limits for JIT-friendly arrays.
MAX_PLANETS = 96
MAX_FLEETS = 256
MAX_COMET_GROUPS = 8
MAX_COMET_PATH_LEN = 64
MAX_COMET_PLANETS = MAX_COMET_GROUPS * 4
MAX_MOVES_PER_PLAYER = 48
NUM_PLAYERS = 2

PLANET_COLS = 8  # id, owner, x, y, radius, ships, production, active
FLEET_COLS = 8  # id, owner, x, y, angle, from_planet_id, ships, active


In [ ]:
%%writefile src/orbit_wars/step.py
"""JAX simulation step for Orbit Wars (vectorized, vmap-friendly)."""

from __future__ import annotations

import jax
import jax.numpy as jnp
import numpy as np

from .comet import spawn_comet_for_state
from .constants import (
    CENTER,
    COMET_SPAWN_STEPS,
    MAX_COMET_GROUPS,
    MAX_COMET_PATH_LEN,
    MAX_COMET_PLANETS,
    MAX_FLEETS,
    MAX_MOVES_PER_PLAYER,
    MAX_PLANETS,
    NUM_PLAYERS,
    ROTATION_RADIUS_LIMIT,
)
from .convert import pack_comets
from .geometry import fleet_speed, in_bounds, sun_hit, swept_pair_hit
from .state import OrbitWarsState

# ---------------------------------------------------------------------------
# Python-side comet bookkeeping (rare: at most ~10 expiries + 5 spawns per game)
# ---------------------------------------------------------------------------


def remove_expired_comets(state: OrbitWarsState) -> OrbitWarsState:
    """Mark comet planets inactive when their path is exhausted.

    Pure-Python numpy implementation; cheap because it short-circuits when no
    comet groups are active.
    """
    active_groups = np.asarray(state.comets.active)
    if not active_groups.any():
        return state

    planets = np.asarray(state.planets).copy()
    initial = np.asarray(state.initial_planets).copy()
    planet_ids = np.asarray(state.comets.planet_ids)
    path_index = np.asarray(state.comets.path_index)
    path_lengths = np.asarray(state.comets.path_lengths)

    # Build a map planet_id -> slot for active planets once.
    active_mask = planets[:, 7] > 0.0
    id_to_slot = {int(planets[i, 0]): i for i in range(MAX_PLANETS) if active_mask[i]}

    mutated = False
    for gi in range(MAX_COMET_GROUPS):
        if not active_groups[gi]:
            continue
        idx_now = int(path_index[gi])
        for pi in range(4):
            pid = int(planet_ids[gi, pi])
            plen = int(path_lengths[gi, pi])
            if pid < 0 or plen <= 0 or idx_now < plen:
                continue
            slot = id_to_slot.get(pid)
            if slot is None:
                continue
            planets[slot, 7] = 0.0
            initial[slot, 7] = 0.0
            mutated = True

    if not mutated:
        return state
    return state.replace(planets=jnp.asarray(planets), initial_planets=jnp.asarray(initial))


def _maybe_spawn_comet_numpy(state: OrbitWarsState) -> OrbitWarsState:
    next_step = int(state.step) + 1
    if next_step not in COMET_SPAWN_STEPS or bool(state.done):
        return state

    spawn = spawn_comet_for_state(
        np.asarray(state.planets),
        int(state.n_planets),
        np.asarray(state.initial_planets),
        np.asarray(state.comet_planet_ids),
        int(state.n_comet_planet_ids),
        float(state.angular_velocity),
        next_step,
        int(state.episode_seed),
        comet_speed=4.0,
    )
    if spawn is None:
        return state

    planets = np.asarray(state.planets).copy()
    initial = np.asarray(state.initial_planets).copy()
    n = int(state.n_planets)
    for row in spawn["new_planets"]:
        if n >= MAX_PLANETS:
            break
        planets[n, :7] = row
        planets[n, 7] = 1.0
        initial[n, :7] = row
        initial[n, 7] = 1.0
        n += 1

    comet_ids = np.asarray(state.comet_planet_ids).copy()
    nc = int(state.n_comet_planet_ids)
    for pid in spawn["new_comet_ids"]:
        if nc >= MAX_COMET_PLANETS:
            break
        comet_ids[nc] = int(pid)
        nc += 1

    from .convert import _unpack_comets

    comets_list = _unpack_comets(state.comets)
    comets_list.append(spawn["group"])
    comets = pack_comets(comets_list)

    return state.replace(
        planets=jnp.asarray(planets),
        initial_planets=jnp.asarray(initial),
        n_planets=jnp.int32(n),
        comets=comets,
        comet_planet_ids=jnp.asarray(comet_ids),
        n_comet_planet_ids=jnp.int32(nc),
    )


# ---------------------------------------------------------------------------
# Vectorized JIT physics
# ---------------------------------------------------------------------------


def _apply_moves(
    state: OrbitWarsState,
    actions: jnp.ndarray,
    action_mask: jnp.ndarray,
    player: jnp.int32,
) -> OrbitWarsState:
    """Apply one player's moves sequentially."""
    planets = state.planets
    fleets = state.fleets
    n_fleets = state.n_fleets
    next_fleet_id = state.next_fleet_id

    pids_i32 = planets[:, 0].astype(jnp.int32)
    active_planet = planets[:, 7] > 0.0

    move_from = actions[:, 0].astype(jnp.int32)
    move_angle = actions[:, 1]
    move_ships_i32 = actions[:, 2].astype(jnp.int32)

    match = (move_from[:, None] == pids_i32[None, :]) & active_planet[None, :]
    source_idx = jnp.argmax(match.astype(jnp.int32), axis=-1)
    source_exists = jnp.any(match, axis=-1)

    use_move = (action_mask > 0.0) & (move_ships_i32 > 0) & source_exists

    player_f = player.astype(jnp.float32)

    def body(i, carry):
        planets_c, fleets_c, n_fleets_c, next_id_c = carry
        sidx = source_idx[i]
        safe_sidx = jnp.maximum(sidx, 0)
        owner = planets_c[safe_sidx, 1]
        have = planets_c[safe_sidx, 5].astype(jnp.int32)
        ships = move_ships_i32[i]
        valid = use_move[i] & (owner == player_f) & (have >= ships)

        new_ship_count = planets_c[safe_sidx, 5] - ships.astype(jnp.float32)
        planets_c = planets_c.at[safe_sidx, 5].set(
            jnp.where(valid, new_ship_count, planets_c[safe_sidx, 5])
        )
        radius = planets_c[safe_sidx, 4]
        start_x = planets_c[safe_sidx, 2] + jnp.cos(move_angle[i]) * (radius + 0.1)
        start_y = planets_c[safe_sidx, 3] + jnp.sin(move_angle[i]) * (radius + 0.1)

        slot = n_fleets_c
        can_add = valid & (slot < MAX_FLEETS)
        safe_slot = jnp.minimum(slot, MAX_FLEETS - 1)
        new_row = jnp.array(
            [
                next_id_c.astype(jnp.float32),
                player_f,
                start_x,
                start_y,
                move_angle[i],
                move_from[i].astype(jnp.float32),
                ships.astype(jnp.float32),
                1.0,
            ],
            dtype=jnp.float32,
        )
        fleets_c = fleets_c.at[safe_slot].set(
            jnp.where(can_add, new_row, fleets_c[safe_slot])
        )
        n_fleets_c = jnp.where(can_add, n_fleets_c + 1, n_fleets_c)
        next_id_c = jnp.where(can_add, next_id_c + 1, next_id_c)
        return (planets_c, fleets_c, n_fleets_c, next_id_c)

    planets, fleets, n_fleets, next_fleet_id = jax.lax.fori_loop(
        0, MAX_MOVES_PER_PLAYER, body, (planets, fleets, n_fleets, next_fleet_id),
    )
    return state.replace(
        planets=planets, fleets=fleets, n_fleets=n_fleets, next_fleet_id=next_fleet_id,
    )


def _production(state: OrbitWarsState) -> OrbitWarsState:
    owned = (state.planets[:, 1] >= 0.0) & (state.planets[:, 7] > 0.0)
    planets = state.planets.at[:, 5].set(
        jnp.where(owned, state.planets[:, 5] + state.planets[:, 6], state.planets[:, 5])
    )
    return state.replace(planets=planets)


def _compute_planet_paths(state: OrbitWarsState):
    """Returns (pids, old_x, old_y, new_x, new_y, radius, check)."""
    pids_i32 = state.planets[:, 0].astype(jnp.int32)
    old_x = state.planets[:, 2]
    old_y = state.planets[:, 3]
    radius = state.planets[:, 4]
    active = state.planets[:, 7] > 0.0

    cpids = state.comet_planet_ids
    valid_cpid = cpids >= 0
    is_comet = active & jnp.any(
        (pids_i32[:, None] == cpids[None, :]) & valid_cpid[None, :], axis=-1
    )

    init = state.initial_planets
    dx0 = init[:, 2] - CENTER
    dy0 = init[:, 3] - CENTER
    orbit_r = jnp.sqrt(dx0 * dx0 + dy0 * dy0)
    rotating = active & (orbit_r + radius < ROTATION_RADIUS_LIMIT) & (~is_comet)
    initial_angle = jnp.arctan2(dy0, dx0)
    current_angle = initial_angle + state.angular_velocity * state.step.astype(jnp.float32)
    rot_x = CENTER + orbit_r * jnp.cos(current_angle)
    rot_y = CENTER + orbit_r * jnp.sin(current_angle)
    new_x = jnp.where(rotating, rot_x, old_x)
    new_y = jnp.where(rotating, rot_y, old_y)
    check = jnp.where(active & (~is_comet), 1.0, 0.0)

    # ---- Comet path lookup ------------------------
    comets = state.comets
    cgpids = comets.planet_ids
    cplens = comets.path_lengths
    cactive = comets.active
    idx_g = comets.path_index + 1
    idx_clip = jnp.clip(idx_g, 0, MAX_COMET_PATH_LEN - 1)

    match_gqp = (
        (cgpids[..., None] == pids_i32[None, None, :])
        & active[None, None, :]
        & (cgpids[..., None] >= 0)
    )
    slot_gq = jnp.argmax(match_gqp.astype(jnp.int32), axis=-1)
    has_match_gq = jnp.any(match_gqp, axis=-1)

    on_board_gq = (idx_g[:, None] < cplens) & has_match_gq & cactive[:, None]
    is_comet_slot_gq = has_match_gq & cactive[:, None]

    g_arange = jnp.arange(MAX_COMET_GROUPS)
    q_arange = jnp.arange(4)
    g_grid, q_grid = jnp.meshgrid(g_arange, q_arange, indexing="ij")
    idx_grid = jnp.broadcast_to(idx_clip[:, None], (MAX_COMET_GROUPS, 4))
    path_xy = comets.paths[g_grid, q_grid, idx_grid, :]
    path_px = path_xy[..., 0]
    path_py = path_xy[..., 1]

    flat_slot = slot_gq.reshape(-1)
    flat_on = on_board_gq.reshape(-1)
    flat_px = path_px.reshape(-1)
    flat_py = path_py.reshape(-1)
    flat_is_comet_slot = is_comet_slot_gq.reshape(-1)

    new_x = new_x.at[flat_slot].set(jnp.where(flat_on, flat_px, new_x[flat_slot]))
    new_y = new_y.at[flat_slot].set(jnp.where(flat_on, flat_py, new_y[flat_slot]))

    # FIX: Newly placed comets (idx_g == 0) are not collision-eligible (Point 3).
    flat_idx_g = jnp.broadcast_to(idx_g[:, None], (MAX_COMET_GROUPS, 4)).reshape(-1)
    check_at_slot = jnp.where(flat_is_comet_slot & (flat_idx_g > 0), 1.0, check[flat_slot])
    check = check.at[flat_slot].set(check_at_slot)

    return pids_i32, old_x, old_y, new_x, new_y, radius, check


def _move_fleets(state, pids_i32, old_x, old_y, new_x, new_y, radius, check):
    """Returns (state, combat[planet, player])."""
    max_speed = state.ship_speed
    fleets = state.fleets

    active_f = fleets[:, 7] > 0.0
    owner = fleets[:, 1].astype(jnp.int32)
    angle = fleets[:, 4]
    ships = fleets[:, 6]
    old_fx = fleets[:, 2]
    old_fy = fleets[:, 3]
    speed = fleet_speed(ships, max_speed)
    new_fx = old_fx + jnp.cos(angle) * speed
    new_fy = old_fy + jnp.sin(angle) * speed

    hit_fp = swept_pair_hit(
        old_fx[:, None], old_fy[:, None], new_fx[:, None], new_fy[:, None],
        old_x[None, :], old_y[None, :], new_x[None, :], new_y[None, :], radius[None, :],
    )
    eligible = (check[None, :] > 0.0) & active_f[:, None]
    hit_fp = hit_fp & eligible

    any_hit = jnp.any(hit_fp, axis=-1)
    first_hit = jnp.argmax(hit_fp.astype(jnp.int32), axis=-1)
    hit_idx = jnp.where(any_hit, first_hit, -1)

    out_bounds = ~in_bounds(new_fx, new_fy)
    sun = sun_hit(old_fx, old_fy, new_fx, new_fy)
    remove = active_f & ((hit_idx >= 0) | out_bounds | sun)

    combat = jnp.zeros((MAX_PLANETS, NUM_PLAYERS), dtype=jnp.float32)
    hit_mask = active_f & (hit_idx >= 0)
    safe_hit_idx = jnp.where(hit_mask, hit_idx, 0)
    safe_owner = jnp.clip(owner, 0, NUM_PLAYERS - 1)
    contrib = jnp.where(hit_mask, ships, 0.0)
    combat = combat.at[safe_hit_idx, safe_owner].add(contrib)

    fleets = fleets.at[:, 2].set(new_fx)
    fleets = fleets.at[:, 3].set(new_fy)
    fleets = fleets.at[:, 7].set(jnp.where(remove, 0.0, fleets[:, 7]))
    return state.replace(fleets=fleets), combat


def _apply_planet_positions(state, new_x, new_y) -> OrbitWarsState:
    planets = state.planets.at[:, 2].set(new_x)
    planets = planets.at[:, 3].set(new_y)
    return state.replace(planets=planets)


def _advance_comet_indices(state: OrbitWarsState) -> OrbitWarsState:
    comets = state.comets
    comets = comets.replace(path_index=comets.path_index + 1)
    return state.replace(comets=comets)


def _expire_comets_in_jit(state: OrbitWarsState) -> OrbitWarsState:
    """Deactivate comet planets whose path has ended (vectorized, JIT-safe)."""
    comets = state.comets
    cgpids = comets.planet_ids
    cplens = comets.path_lengths
    cactive = comets.active
    idx_now = comets.path_index

    pids_i32 = state.planets[:, 0].astype(jnp.int32)
    active = state.planets[:, 7] > 0.0

    # FIX: Expire if the NEXT index would be out of bounds (Point 2)
    expired_gq = (idx_now[:, None] + 1 >= cplens) & (cplens > 0) & cactive[:, None] & (cgpids >= 0)
    match_gqp = (
        (cgpids[..., None] == pids_i32[None, None, :])
        & active[None, None, :]
        & (cgpids[..., None] >= 0)
    )
    slot_gq = jnp.argmax(match_gqp.astype(jnp.int32), axis=-1)
    has_match_gq = jnp.any(match_gqp, axis=-1)
    do_expire = (expired_gq & has_match_gq).reshape(-1)
    flat_slot = slot_gq.reshape(-1)

    planets = state.planets
    new_active = jnp.where(do_expire, 0.0, planets[flat_slot, 7])
    planets = planets.at[flat_slot, 7].set(new_active)
    initial = state.initial_planets
    new_active_i = jnp.where(do_expire, 0.0, initial[flat_slot, 7])
    initial = initial.at[flat_slot, 7].set(new_active_i)
    return state.replace(planets=planets, initial_planets=initial)


def _resolve_combat(state: OrbitWarsState, combat: jnp.ndarray) -> OrbitWarsState:
    planets = state.planets
    active = planets[:, 7] > 0.0
    ships_p0 = combat[:, 0]
    ships_p1 = combat[:, 1]
    total = ships_p0 + ships_p1
    contested = active & (total > 0.0)

    top = jnp.maximum(ships_p0, ships_p1)
    second = jnp.minimum(ships_p0, ships_p1)
    tie = ships_p0 == ships_p1
    survivor_ships = jnp.where(tie, 0.0, top - second)
    top_player = jnp.where(ships_p0 >= ships_p1, 0, 1).astype(jnp.int32)
    survivor_owner = jnp.where(survivor_ships > 0.0, top_player, -1)

    owner = planets[:, 1].astype(jnp.int32)
    same = owner == survivor_owner
    diff = (~same) & (survivor_owner >= 0)

    new_ships_same = planets[:, 5] + survivor_ships
    new_ships_diff = planets[:, 5] - survivor_ships
    captured = new_ships_diff < 0.0
    new_ships = jnp.where(
        same,
        new_ships_same,
        jnp.where(captured, jnp.abs(new_ships_diff), new_ships_diff),
    )
    new_owner = jnp.where(
        contested & diff & captured,
        survivor_owner.astype(jnp.float32),
        planets[:, 1],
    )
    planets = planets.at[:, 5].set(jnp.where(contested, new_ships, planets[:, 5]))
    planets = planets.at[:, 1].set(jnp.where(contested & diff, new_owner, planets[:, 1]))
    return state.replace(planets=planets)


def _termination(state: OrbitWarsState) -> OrbitWarsState:
    step = state.step
    terminated_by_steps = step >= (state.episode_steps - 1)

    owned = (state.planets[:, 1] >= 0.0) & (state.planets[:, 7] > 0.0)
    fleet_active = state.fleets[:, 7] > 0.0
    fleet_owners = state.fleets[:, 1].astype(jnp.int32)
    planet_owners = state.planets[:, 1].astype(jnp.int32)

    p_range = jnp.arange(NUM_PLAYERS, dtype=jnp.int32)
    owned_match = owned[None, :] & (planet_owners[None, :] == p_range[:, None])
    fleet_match = fleet_active[None, :] & (fleet_owners[None, :] == p_range[:, None])

    has_any = jnp.any(owned_match, axis=-1) | jnp.any(fleet_match, axis=-1)
    alive_count = jnp.sum(has_any.astype(jnp.int32))
    terminated = terminated_by_steps | (alive_count <= 1)

    planet_score = jnp.sum(jnp.where(owned_match, state.planets[None, :, 5], 0.0), axis=-1)
    fleet_score = jnp.sum(jnp.where(fleet_match, state.fleets[None, :, 6], 0.0), axis=-1)
    scores = planet_score + fleet_score

    max_score = jnp.max(scores)
    all_max = jnp.all(scores == max_score)
    
    rewards = jnp.where(
        terminated,
        jnp.where(
            all_max | (max_score <= 0.0),
            jnp.zeros((NUM_PLAYERS,), dtype=jnp.float32),
            jnp.where(scores == max_score, 1.0, -1.0)
        ),
        jnp.zeros((NUM_PLAYERS,), dtype=jnp.float32),
    )
    return state.replace(done=terminated, rewards=rewards)


# ---------------------------------------------------------------------------
# Public step API
# ---------------------------------------------------------------------------


@jax.jit
def step_jit(
    state: OrbitWarsState,
    actions_p0: jnp.ndarray,
    actions_p1: jnp.ndarray,
    mask_p0: jnp.ndarray,
    mask_p1: jnp.ndarray,
) -> OrbitWarsState:
    # 1. Expire comets before processing moves (Point 2)
    state = _expire_comets_in_jit(state)
    
    state = _apply_moves(state, actions_p0, mask_p0, jnp.int32(0))
    state = _apply_moves(state, actions_p1, mask_p1, jnp.int32(1))
    state = _production(state)
    pids, old_x, old_y, new_x, new_y, radius, check = _compute_planet_paths(state)
    state, combat = _move_fleets(state, pids, old_x, old_y, new_x, new_y, radius, check)
    state = _apply_planet_positions(state, new_x, new_y)
    
    # 2. Advance indices for the NEXT step
    state = _advance_comet_indices(state)
    
    state = _resolve_combat(state, combat)
    state = state.replace(step=state.step + 1)
    state = _termination(state)
    return state


def step(
    state: OrbitWarsState,
    actions: list[list[list[float | int]]] | None = None,
    *,
    actions_p0: jnp.ndarray | None = None,
    actions_p1: jnp.ndarray | None = None,
    mask_p0: jnp.ndarray | None = None,
    mask_p1: jnp.ndarray | None = None,
) -> OrbitWarsState:
    state = _maybe_spawn_comet_numpy(state)

    if actions is not None:
        a0, m0 = _list_action_to_padded(actions[0])
        a1, m1 = _list_action_to_padded(actions[1])
    else:
        a0, m0 = actions_p0, mask_p0
        a1, m1 = actions_p1, mask_p1

    state = step_jit(state, a0, a1, m0, m1)
    return state


def _list_action_to_padded(moves: list[list[float | int]]) -> tuple[jnp.ndarray, jnp.ndarray]:
    arr = np.zeros((MAX_MOVES_PER_PLAYER, 3), dtype=np.float32)
    mask = np.zeros((MAX_MOVES_PER_PLAYER,), dtype=np.float32)
    n = min(len(moves), MAX_MOVES_PER_PLAYER)
    for i in range(n):
        move = moves[i]
        if len(move) != 3:
            continue
        arr[i, 0] = float(move[0])
        arr[i, 1] = float(move[1])
        arr[i, 2] = float(move[2])
        mask[i] = 1.0
    return jnp.asarray(arr), jnp.asarray(mask)


@jax.jit
def batched_step(
    states: OrbitWarsState,
    actions_p0: jnp.ndarray,
    actions_p1: jnp.ndarray,
    mask_p0: jnp.ndarray,
    mask_p1: jnp.ndarray,
) -> OrbitWarsState:
    return jax.vmap(step_jit)(states, actions_p0, actions_p1, mask_p0, mask_p1)


In [ ]:
%%writefile src/orbit_wars/comet.py
"""Comet spawn logic (vendored from official orbit_wars.py — no kaggle import needed)."""

from __future__ import annotations

import math
import random
from typing import Any

import numpy as np

from .constants import (
    BOARD_SIZE,
    CENTER,
    COMET_PRODUCTION,
    COMET_RADIUS,
    ROTATION_RADIUS_LIMIT,
    SUN_RADIUS,
)


def _distance(p1: tuple[float, float], p2: tuple[float, float]) -> float:
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def generate_comet_paths(
    initial_planets: list[list[float]],
    angular_velocity: float,
    spawn_step: int,
    comet_planet_ids: list[int] | set[int] | None = None,
    comet_speed: float = 4.0,
    rng: random.Random | None = None,
) -> list[list[list[float]]] | None:
    """Generate 4 symmetric elliptical comet paths (matches official env)."""
    if rng is None:
        rng = random.Random()
    comet_ids = set(comet_planet_ids or [])

    for _ in range(300):
        e = rng.uniform(0.75, 0.93)
        a = rng.uniform(60, 150)
        perihelion = a * (1 - e)
        if perihelion < SUN_RADIUS + COMET_RADIUS:
            continue

        b = a * math.sqrt(1 - e**2)
        c_val = a * e
        phi = rng.uniform(math.pi / 6, math.pi / 3)

        dense: list[tuple[float, float]] = []
        num = 5000
        for i in range(num):
            t = 0.3 * math.pi + 1.4 * math.pi * i / (num - 1)
            ex = c_val + a * math.cos(t)
            ey = b * math.sin(t)
            x = CENTER + ex * math.cos(phi) - ey * math.sin(phi)
            y = CENTER + ex * math.sin(phi) + ey * math.cos(phi)
            dense.append((x, y))

        path = [dense[0]]
        cum = 0.0
        target = comet_speed
        for i in range(1, len(dense)):
            cum += _distance(dense[i], dense[i - 1])
            if cum >= target:
                path.append(dense[i])
                target += comet_speed

        board_start = None
        board_end = None
        for i, (x, y) in enumerate(path):
            if 0 <= x <= BOARD_SIZE and 0 <= y <= BOARD_SIZE:
                if board_start is None:
                    board_start = i
                board_end = i

        if board_start is None:
            continue
        visible = path[board_start : board_end + 1]
        if not (5 <= len(visible) <= 40):
            continue

        paths = [
            [[y, x] for x, y in visible],
            [[BOARD_SIZE - x, y] for x, y in visible],
            [[x, BOARD_SIZE - y] for x, y in visible],
            [[BOARD_SIZE - y, BOARD_SIZE - x] for x, y in visible],
        ]

        static_planets: list[list[float]] = []
        orbiting_planets: list[list[float]] = []
        for planet in initial_planets:
            if planet[0] in comet_ids:
                continue
            pr = _distance((planet[2], planet[3]), (CENTER, CENTER))
            if pr + planet[4] < ROTATION_RADIUS_LIMIT:
                orbiting_planets.append(planet)
            else:
                static_planets.append(planet)

        valid = True
        buf = COMET_RADIUS + 0.5
        for k, (cx, cy) in enumerate(visible):
            if _distance((cx, cy), (CENTER, CENTER)) < SUN_RADIUS + COMET_RADIUS:
                valid = False
                break

            sym_pts = [
                (cy, cx),
                (BOARD_SIZE - cx, cy),
                (cx, BOARD_SIZE - cy),
                (BOARD_SIZE - cy, BOARD_SIZE - cx),
            ]
            for planet in static_planets:
                for sp in sym_pts:
                    if _distance(sp, (planet[2], planet[3])) < planet[4] + buf:
                        valid = False
                        break
                if not valid:
                    break
            if not valid:
                break

            game_step = spawn_step - 1 + k
            for planet in orbiting_planets:
                dx = planet[2] - CENTER
                dy = planet[3] - CENTER
                orb_r = math.sqrt(dx**2 + dy**2)
                init_angle = math.atan2(dy, dx)
                cur_angle = init_angle + angular_velocity * game_step
                px = CENTER + orb_r * math.cos(cur_angle)
                py = CENTER + orb_r * math.sin(cur_angle)
                for sp in sym_pts:
                    if _distance(sp, (px, py)) < planet[4] + COMET_RADIUS:
                        valid = False
                        break
                if not valid:
                    break
            if not valid:
                break

        if valid:
            return paths
    return None


def spawn_comet_for_state(
    planets: np.ndarray,
    n_planets: int,
    initial_planets: np.ndarray,
    comet_planet_ids: np.ndarray,
    n_comet_ids: int,
    angular_velocity: float,
    spawn_step: int,
    episode_seed: int,
    comet_speed: float = 4.0,
) -> dict[str, Any] | None:
    """Return dict with new planet rows + comet group, or None if spawn fails."""
    planet_rows = [
        [float(planets[i, j]) for j in range(7)]
        for i in range(int(n_planets))
        if planets[i, 7] > 0.0
    ]
    initial_rows = [
        [float(initial_planets[i, j]) for j in range(7)]
        for i in range(int(n_planets))
        if initial_planets[i, 7] > 0.0
    ]
    comet_ids = [int(comet_planet_ids[i]) for i in range(int(n_comet_ids)) if int(comet_planet_ids[i]) >= 0]
    comet_rng = random.Random(f"orbit_wars-comet-{episode_seed}-{spawn_step}")
    paths = generate_comet_paths(
        initial_rows,
        float(angular_velocity),
        int(spawn_step),
        comet_ids,
        float(comet_speed),
        rng=comet_rng,
    )
    if not paths:
        return None

    next_id = max(int(p[0]) for p in planet_rows) + 1
    comet_ships = min(
        comet_rng.randint(1, 99),
        comet_rng.randint(1, 99),
        comet_rng.randint(1, 99),
        comet_rng.randint(1, 99),
    )
    new_planets: list[list[float]] = []
    group_pids: list[int] = []
    for i, _path in enumerate(paths):
        pid = next_id + i
        group_pids.append(pid)
        new_planets.append([pid, -1, -99.0, -99.0, COMET_RADIUS, comet_ships, COMET_PRODUCTION])
    return {
        "new_planets": new_planets,
        "group": {"planet_ids": group_pids, "paths": paths, "path_index": -1},
        "new_comet_ids": group_pids,
    }


In [ ]:
%%writefile src/orbit_wars/decode.py
"""Action decoding and grid composition for Orbit Wars.
Splits composition into two phases (Target Phase and Bucket Phase) to avoid
expensive O(P*P*B) calculations and huge intermediate tensors.
"""

from __future__ import annotations

import jax
import jax.numpy as jnp

from .constants import (
    MIN_LAUNCH_SHIPS,
    PATH_PLANET_MARGIN,
    SUN_PATH_MARGIN,
    INTERCEPT_ITERATIONS,
)
from .geometry import (
    is_orbiting_planet,
    point_to_segment_distance,
    estimate_intercept_angles,
)
from .state import OrbitWarsState


def ship_counts_for_buckets(
    src_ships: jnp.ndarray, # (...)
    tgt_ships: jnp.ndarray, # (...)
    inc_me: jnp.ndarray,    # (...)
    inc_en: jnp.ndarray,    # (...)
) -> jnp.ndarray:
    """Return an array of ship counts for 4 buckets: 25%, 50%, 75%, 100%."""
    b0 = src_ships * 0.25
    b1 = src_ships * 0.50
    b2 = src_ships * 0.75
    b3 = src_ships * 1.00
    
    counts = jnp.stack([b0, b1, b2, b3], axis=-1)
    # Floor at MIN_LAUNCH_SHIPS to prevent 1-ship spamming.
    # We clip to at least MIN_LAUNCH_SHIPS, but bucket_validity_mask will handle the logic
    # of ensuring we actually HAVE that many ships to send.
    from .constants import MIN_LAUNCH_SHIPS
    return jnp.clip(jnp.floor(counts), float(MIN_LAUNCH_SHIPS), jnp.maximum(float(MIN_LAUNCH_SHIPS), src_ships[..., None]))


def bucket_validity_mask(
    ship_counts: jnp.ndarray, source_ships: jnp.ndarray
) -> jnp.ndarray:
    src = source_ships[..., None]
    has_enough_to_launch = src >= jnp.float32(MIN_LAUNCH_SHIPS)
    return has_enough_to_launch & (ship_counts > 0.0) & (ship_counts <= src)


def path_blocked_by_planets(
    start_x: jnp.ndarray,
    start_y: jnp.ndarray,
    target_x: jnp.ndarray,
    target_y: jnp.ndarray,
    planet_x: jnp.ndarray,
    planet_y: jnp.ndarray,
    planet_radius: jnp.ndarray,
    planet_active: jnp.ndarray,
    margin: float = PATH_PLANET_MARGIN,
) -> jnp.ndarray:
    ox = planet_x
    oy = planet_y
    obs_r = planet_radius + margin
    is_obstacle = planet_active

    for _ in range(start_x.ndim):
        ox = ox[None, ...]
        oy = oy[None, ...]
        obs_r = obs_r[None, ...]
        is_obstacle = is_obstacle[None, ...]

    sx = start_x[..., None]
    sy = start_y[..., None]
    tx = target_x[..., None]
    ty = target_y[..., None]

    d = point_to_segment_distance(ox, oy, sx, sy, tx, ty)
    
    # Ignore obstacles at the exact start or end point (self)
    d_start = jnp.sqrt((ox - sx)**2 + (oy - sy)**2)
    d_target = jnp.sqrt((ox - tx)**2 + (oy - ty)**2)
    is_not_self = (d_start > 1e-3) & (d_target > 1e-3)
    
    return jnp.any((d <= obs_r) & is_obstacle & is_not_self, axis=-1)


def compose_target_grid(
    state: OrbitWarsState,
    player: jnp.int32 | int,
    incoming_me: jnp.ndarray,
    incoming_enemy: jnp.ndarray,
    *,
    intercept_iterations: int = INTERCEPT_ITERATIONS,
    sun_path_margin: float = SUN_PATH_MARGIN,
    path_planet_margin: float = PATH_PLANET_MARGIN,
    enable_planet_block: bool = True,
) -> dict[str, jnp.ndarray]:
    """Phase 1: Compute which (source, target) pairs are potentially valid."""
    planets = state.planets
    active = planets[:, 7] > 0.0
    owner = planets[:, 1]
    player_f = jnp.float32(player)
    source_valid = active & (owner == player_f)
    target_valid = active

    x, y, radius, ships, pids = planets[:, 2], planets[:, 3], planets[:, 4], planets[:, 5], planets[:, 0].astype(jnp.int32)

    tgt_orbiting = is_orbiting_planet(x, y, radius)
    # Representative ships (P,)
    rep_ships = jnp.maximum(1.0, jnp.floor(ships * 0.5))
    
    from .geometry import precompute_comet_trajectories
    is_comet, trajectories, valid_time = precompute_comet_trajectories(
        state.comets.active, state.comets.planet_ids, state.comets.path_index,
        state.comets.paths, state.comets.path_lengths, pids
    )
    
    def _row_intercept(sx, sy, sr):
        # We want to check all targets (P,) for this one source.
        # Outputs (P,) tensors.
        return estimate_intercept_angles(
            sx, sy, sr, x, y, radius, tgt_orbiting, is_comet, 
            trajectories, valid_time, rep_ships,
            state.angular_velocity, state.ship_speed,
            n_iter=intercept_iterations, sun_margin=sun_path_margin,
        )

    angle, aim_x, aim_y, sun_blocks = jax.vmap(_row_intercept)(x, y, radius)

    if enable_planet_block:
        planet_blocks = path_blocked_by_planets(
            x[:, None], y[:, None], x[None, :], y[None, :], x, y, radius, active, margin=path_planet_margin,
        )
    else:
        planet_blocks = jnp.zeros((planets.shape[0], planets.shape[0]), dtype=jnp.bool_)

    pair_valid = source_valid[:, None] & target_valid[None, :]
    has_enough = (ships >= jnp.float32(MIN_LAUNCH_SHIPS))
    target_mask = pair_valid & has_enough[:, None] & (~sun_blocks) & (~planet_blocks)

    return {
        "source_valid_any": jnp.any(target_mask, axis=-1),
        "target_mask": target_mask,
        "from_ids": pids,
        "is_comet": is_comet,
        "trajectories": trajectories,
        "valid_time": valid_time,
        "incoming_me": incoming_me,
        "incoming_enemy": incoming_enemy,
    }


def compose_bucket_grid(
    state: OrbitWarsState,
    target_idx: jnp.ndarray, # (P,) chosen target per source
    phase1_results: dict,
    *,
    intercept_iterations: int = INTERCEPT_ITERATIONS,
    sun_path_margin: float = SUN_PATH_MARGIN,
    **_kwargs,
) -> dict[str, jnp.ndarray]:
    """Phase 2: Compute exact angles and bucket validity for CHOSEN targets."""
    planets = state.planets
    x, y, radius, ships = planets[:, 2], planets[:, 3], planets[:, 4], planets[:, 5]
    
    tx = x[target_idx]
    ty = y[target_idx]
    tr = radius[target_idx]
    torb = is_orbiting_planet(tx, ty, tr)
    tcom = phase1_results["is_comet"][target_idx]
    ttraj = phase1_results["trajectories"][target_idx]
    tvt = phase1_results["valid_time"][target_idx]
    
    tgt_ships = ships[target_idx]
    inc_me = phase1_results["incoming_me"][target_idx]
    inc_en = phase1_results["incoming_enemy"][target_idx]
    
    # ship_counts: (P, B)
    ship_counts = ship_counts_for_buckets(ships, tgt_ships, inc_me, inc_en)
    
    def _bucket_intercept(sx, sy, sr, tx, ty, tr, torb, tcom, ttraj, tvt, sc):
        # All inputs are scalars except sc which is (B,)
        # Returns (B,) tensors.
        return estimate_intercept_angles(
            sx, sy, sr, tx, ty, tr, torb, tcom, ttraj, tvt, sc,
            state.angular_velocity, state.ship_speed,
            n_iter=intercept_iterations, sun_margin=sun_path_margin,
        )

    angle, aim_x, aim_y, sun_blocks = jax.vmap(_bucket_intercept)(
        x, y, radius, tx, ty, tr, torb, tcom, ttraj, tvt, ship_counts
    )
    
    bucket_valid = bucket_validity_mask(ship_counts, ships) & (~sun_blocks)
    
    return {
        "angle": angle,
        "ship_counts": ship_counts,
        "bucket_valid": bucket_valid,
    }


def compose_full_grid(
    state: OrbitWarsState,
    player: jnp.int32 | int,
    incoming_me: jnp.ndarray | None = None,
    incoming_enemy: jnp.ndarray | None = None,
    *,
    intercept_iterations: int = INTERCEPT_ITERATIONS,
    sun_path_margin: float = SUN_PATH_MARGIN,
    path_planet_margin: float = PATH_PLANET_MARGIN,
    enable_planet_block: bool = True,
) -> dict[str, jnp.ndarray]:
    """Compatibility: Builds the full (P, P, B) grid. SLOW."""
    planets = state.planets
    active = planets[:, 7] > 0.0
    owner = planets[:, 1]
    player_f = jnp.float32(player)
    source_valid = active & (owner == player_f)
    target_valid = active
    x, y, radius, ships, pids = planets[:, 2], planets[:, 3], planets[:, 4], planets[:, 5], planets[:, 0].astype(jnp.int32)

    if incoming_me is None or incoming_enemy is None:
        from .features_jax import _fleet_projections
        incoming_me, incoming_enemy, _, _ = _fleet_projections(state, player_f)

    # (P, P, B)
    ship_counts = ship_counts_for_buckets(ships[:, None], ships[None, :], incoming_me[None, :], incoming_enemy[None, :])

    from .geometry import precompute_comet_trajectories
    is_comet, trajectories, valid_time = precompute_comet_trajectories(
        state.comets.active, state.comets.planet_ids, state.comets.path_index,
        state.comets.paths, state.comets.path_lengths, pids
    )
    tgt_orbiting = is_orbiting_planet(x, y, radius)

    def _full_intercept(sx, sy, sr, sc_row):
        # sx, sy, sr are scalars. sc_row is (P, B).
        # We want to check all targets (P,) with their buckets (B,).
        # Returns (P, B) tensors.
        return estimate_intercept_angles(
            sx, sy, sr, x, y, radius, tgt_orbiting, is_comet,
            trajectories, valid_time, sc_row,
            state.angular_velocity, state.ship_speed,
            n_iter=intercept_iterations, sun_margin=sun_path_margin,
        )

    angle, aim_x, aim_y, sun_blocks = jax.vmap(_full_intercept)(x, y, radius, ship_counts)

    if enable_planet_block:
        planet_blocks = path_blocked_by_planets(x[:, None], y[:, None], x[None, :], y[None, :], x, y, radius, active, margin=path_planet_margin)
        planet_blocks = planet_blocks[:, :, None]
    else:
        planet_blocks = jnp.zeros((planets.shape[0], planets.shape[0], 1), dtype=jnp.bool_)

    pair_valid = source_valid[:, None] & target_valid[None, :]
    bucket_valid = bucket_validity_mask(ship_counts, ships)
    full_valid = pair_valid[..., None] & bucket_valid & (~sun_blocks) & (~planet_blocks)

    return {
        "source_valid": source_valid,
        "target_valid": target_valid,
        "pair_valid": pair_valid,
        "bucket_valid": bucket_valid,
        "sun_blocks": sun_blocks,
        "planet_blocks": planet_blocks,
        "full_valid": full_valid,
        "from_ids": pids,
        "angle": angle,
        "ship_counts": ship_counts,
    }

def compose_action_grid(state, player, **kwargs):
    return compose_full_grid(state, player, **kwargs)

def pack_action_row(from_id, angle, ships, valid):
    row = jnp.stack([from_id.astype(jnp.float32), angle.astype(jnp.float32), jnp.floor(ships).astype(jnp.float32)], axis=-1)
    valid_f = valid.astype(jnp.float32)
    return row * valid_f[..., None], valid_f

def launch_angle(src_x, src_y, tgt_x, tgt_y):
    return jnp.arctan2(tgt_y - src_y, tgt_x - src_x)

def path_crosses_sun(src_x, src_y, tgt_x, tgt_y, margin=SUN_PATH_MARGIN):
    from .geometry import sun_hit
    return sun_hit(src_x, src_y, tgt_x, tgt_y, margin=margin)


In [ ]:
%%writefile src/orbit_wars/heuristic_opponent.py
"""Load the frozen kaggle700 heuristic and produce padded action tensors."""

from __future__ import annotations

import importlib.util
import sys
from pathlib import Path
from typing import Any, Callable

import jax.numpy as jnp
import numpy as np

from .constants import MAX_MOVES_PER_PLAYER
from .convert import state_to_observation_dict
from .state import OrbitWarsState


def default_heuristic_path() -> Path:
    """Resolve `versions/kaggle700_current_heuristic/main.py` from repo root."""
    here = Path(__file__).resolve()
    for root in here.parents:
        candidate = root / "versions" / "kaggle700_current_heuristic" / "main.py"
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not find versions/kaggle700_current_heuristic/main.py. "
        "Include `versions/` in your Kaggle dataset or git clone."
    )


def _resolve_custom_path(path: Path) -> Path:
    """Resolve a user-supplied path relative to cwd or repo root."""
    if path.is_absolute():
        return path
    if path.exists():
        return path.resolve()
    here = Path(__file__).resolve()
    for root in here.parents:
        candidate = root / path
        if candidate.exists():
            return candidate
    return path


def load_heuristic_agent(path: Path | None = None) -> Callable[[Any], list]:
    """Import and return the heuristic `agent(obs)` function."""
    if path is None:
        bot_path = default_heuristic_path().resolve()
    else:
        bot_path = _resolve_custom_path(Path(path)).resolve()
    heur_root = bot_path.parent
    if str(heur_root) not in sys.path:
        sys.path.insert(0, str(heur_root))
    spec = importlib.util.spec_from_file_location("heuristic_main", bot_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Failed to load heuristic from {bot_path}")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.agent


def pack_moves_list(moves: list[list[float | int]]) -> tuple[np.ndarray, np.ndarray]:
    """Pack Kaggle-style moves into `(MAX_MOVES, 3)` + mask."""
    actions = np.zeros((MAX_MOVES_PER_PLAYER, 3), dtype=np.float32)
    mask = np.zeros((MAX_MOVES_PER_PLAYER,), dtype=np.float32)
    for i, row in enumerate(moves[:MAX_MOVES_PER_PLAYER]):
        actions[i, 0] = float(row[0])
        actions[i, 1] = float(row[1])
        actions[i, 2] = float(row[2])
        mask[i] = 1.0
    return actions, mask


def heuristic_actions_for_state(state: OrbitWarsState, player: int, agent) -> tuple[np.ndarray, np.ndarray]:
    obs = state_to_observation_dict(state, player=int(player))
    moves = agent(obs)
    return pack_moves_list(moves)


def batched_heuristic_actions(
    states: OrbitWarsState,
    opponent_players: np.ndarray,
    agent,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    """Build padded `(B, M, 3)` action tensors for player 0 and player 1.

    `opponent_players[i]` is the heuristic seat for env *i* (0 or 1).
    Non-heuristic seats are zero with mask 0.
    """
    import jax.tree_util as tu

    n = int(states.step.shape[0])
    a0 = np.zeros((n, MAX_MOVES_PER_PLAYER, 3), dtype=np.float32)
    m0 = np.zeros((n, MAX_MOVES_PER_PLAYER), dtype=np.float32)
    a1 = np.zeros((n, MAX_MOVES_PER_PLAYER, 3), dtype=np.float32)
    m1 = np.zeros((n, MAX_MOVES_PER_PLAYER), dtype=np.float32)

    for i in range(n):
        single = tu.tree_map(lambda x, i=i: x[i], states)
        opp = int(opponent_players[i])
        act, msk = heuristic_actions_for_state(single, opp, agent)
        if opp == 0:
            a0[i] = act
            m0[i] = msk
        else:
            a1[i] = act
            m1[i] = msk

    return jnp.asarray(a0), jnp.asarray(m0), jnp.asarray(a1), jnp.asarray(m1)


In [ ]:
%%writefile src/orbit_wars/rollout.py
"""Sample masked actions from the Transformer policy and pack them for `step_jit`.

Wiring:

    state, params -> features (Phase 1)
                  -> policy out (Phase 2)
                  -> decode target grid (Phase 3a)
                  -> sample target (HERE)
                  -> decode bucket grid for chosen targets (Phase 3b)
                  -> sample bucket (HERE)
                  -> pack (M, 3) action tensor + mask (HERE)

This two-stage decoding avoids the O(P*P*B) bottleneck.
"""

from __future__ import annotations

import functools
import jax
import jax.numpy as jnp

from .constants import MAX_MOVES_PER_PLAYER, INTERCEPT_ITERATIONS, SUN_PATH_MARGIN
from .decode import compose_target_grid, pack_action_row
from .state import OrbitWarsState

_NEG_INF = jnp.float32(-1e9)


def _masked_log_softmax(logits: jnp.ndarray, mask: jnp.ndarray) -> jnp.ndarray:
    """log_softmax with -inf for masked entries."""
    any_valid = jnp.any(mask, axis=-1, keepdims=True)
    safe_logits = jnp.where(mask, logits, _NEG_INF)
    # Replace fully-masked rows with zero logits to avoid -inf log_softmax NaN.
    safe_logits = jnp.where(any_valid, safe_logits, jnp.zeros_like(logits))
    return jax.nn.log_softmax(safe_logits, axis=-1)


def _entropy_from_log_probs(log_probs: jnp.ndarray, mask: jnp.ndarray) -> jnp.ndarray:
    """Sum -p log p over masked entries (axis=-1)."""
    p = jnp.exp(log_probs) * mask.astype(log_probs.dtype)
    return -jnp.sum(p * log_probs, axis=-1)


def sample_actions(
    rng: jax.Array,
    target_logits: jnp.ndarray,    # (B, P, P)
    bucket_logits: jnp.ndarray,    # (B, P, P, BUCKETS)
    state: OrbitWarsState,         # (B,)
    phase1: dict,                  # vmapped output of compose_target_grid
    deterministic: bool = False,
    intercept_iterations: int = INTERCEPT_ITERATIONS,
    sun_path_margin: float = SUN_PATH_MARGIN,
    **kwargs,
) -> dict[str, jnp.ndarray]:
    """Sample (target, bucket) per source planet with split-phase grid."""
    target_mask = phase1["target_mask"]              # (B, P, P)
    source_valid_any = phase1["source_valid_any"]    # (B, P)

    # 1. Target distribution: log_softmax with mask.
    tgt_log_probs = _masked_log_softmax(target_logits, target_mask)
    entropy_target = _entropy_from_log_probs(tgt_log_probs, target_mask)

    # Sample target.
    b, p, _ = target_logits.shape
    rng, k_tgt = jax.random.split(rng)
    tgt_keys = jax.random.split(k_tgt, b * p).reshape(b, p, 2)

    def _sample_target(lp_row, key):
        if deterministic:
            return jnp.argmax(lp_row, axis=-1)
        return jax.random.categorical(key, lp_row, axis=-1)

    target_idx = jax.vmap(jax.vmap(_sample_target))(tgt_log_probs, tgt_keys).astype(jnp.int32)
    # Mask out invalid sources
    target_idx = jnp.where(source_valid_any, target_idx, jnp.int32(0))

    tgt_lp = jnp.take_along_axis(tgt_log_probs, target_idx[..., None], axis=-1).squeeze(-1)
    tgt_lp = jnp.where(source_valid_any, tgt_lp, jnp.float32(0.0))

    # 2. Phase 2: Compute buckets for CHOSEN targets only (O(P*B) instead of O(P*P*B))
    from .decode import compose_bucket_grid
    bucket_grid = jax.vmap(functools.partial(
        compose_bucket_grid, 
        intercept_iterations=intercept_iterations,
        sun_path_margin=sun_path_margin,
        **kwargs,
    ))(state, target_idx, phase1)
    
    chosen_bucket_valid = bucket_grid["bucket_valid"] # (B, P, BUCKETS)
    
    # Gather bucket logits for chosen target: (B, P, BUCKETS)
    bi = jnp.arange(b)[:, None]
    si = jnp.arange(p)[None, :]
    chosen_bucket_logits = bucket_logits[bi, si, target_idx]

    bkt_log_probs = _masked_log_softmax(chosen_bucket_logits, chosen_bucket_valid)
    entropy_bucket = _entropy_from_log_probs(bkt_log_probs, chosen_bucket_valid)

    # 3. Sample bucket.
    rng, k_bkt = jax.random.split(rng)
    bkt_keys = jax.random.split(k_bkt, b * p).reshape(b, p, 2)
    bucket_idx = jax.vmap(jax.vmap(_sample_target))(bkt_log_probs, bkt_keys).astype(jnp.int32)
    bucket_idx = jnp.where(source_valid_any, bucket_idx, jnp.int32(0))

    bkt_lp = jnp.take_along_axis(bkt_log_probs, bucket_idx[..., None], axis=-1).squeeze(-1)
    bkt_lp = jnp.where(source_valid_any, bkt_lp, jnp.float32(0.0))

    return {
        "target_idx": target_idx,
        "bucket_idx": bucket_idx,
        "log_prob": tgt_lp + bkt_lp,
        "entropy_target": entropy_target,
        "entropy_bucket": entropy_bucket,
        "source_valid": source_valid_any,
        "chosen_bucket_valid": chosen_bucket_valid,
        "angle": bucket_grid["angle"],
        "ship_counts": bucket_grid["ship_counts"],
    }


def pack_padded_actions(
    target_idx: jnp.ndarray,        # (B, P)
    bucket_idx: jnp.ndarray,        # (B, P)
    source_valid: jnp.ndarray,      # (B, P)
    from_ids: jnp.ndarray,          # (B, P)
    angle: jnp.ndarray,             # (B, P, B)  -- GATHERED for chosen target
    ship_counts: jnp.ndarray,       # (B, P, B)  -- GATHERED for chosen target
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    """Packs the chosen moves into a (B, 48, 3) action tensor."""
    b, p = target_idx.shape
    
    # Gather chosen angle and ships
    bi = jnp.arange(b)[:, None]
    si = jnp.arange(p)[None, :]
    chosen_angle = angle[bi, si, bucket_idx]
    chosen_ships = ship_counts[bi, si, bucket_idx]
    
    # Identify NOOP moves (target == source)
    s_range = jnp.arange(p)
    is_noop = (target_idx == s_range[None, :])
    env_mask = source_valid & (~is_noop)

    rows, mask_vals = jax.vmap(pack_action_row)(from_ids, chosen_angle, chosen_ships, env_mask)

    # Compact sources so all valid ones are first.
    sort_key = (-mask_vals).astype(jnp.float32)
    sort_idx = jnp.argsort(sort_key, axis=-1)
    rows_sorted = jnp.take_along_axis(rows, sort_idx[..., None].repeat(3, axis=-1), axis=1)
    mask_sorted = jnp.take_along_axis(mask_vals, sort_idx, axis=-1)

    actions = rows_sorted[:, :MAX_MOVES_PER_PLAYER, :]
    action_mask = mask_sorted[:, :MAX_MOVES_PER_PLAYER]

    # executed_mask for PPO. Includes NOOPs so they contribute to entropy and policy gradients.
    rank = jnp.argsort(sort_idx, axis=-1)
    executed_mask = source_valid & (rank < MAX_MOVES_PER_PLAYER)

    return actions, action_mask, executed_mask


def policy_step(
    rng: jax.Array,
    policy_apply,
    params,
    states,                          # vmapped OrbitWarsState
    features: dict,                  # vmapped encoder output
    player_per_env: jnp.ndarray,     # (B,) int32
    deterministic: bool = False,
) -> dict:
    """One full policy step with split-phase grid composition."""
    out = policy_apply(params, **features)
    
    # Phase 1: Target Grid
    phase1 = jax.vmap(compose_target_grid, in_axes=(0, 0, 0, 0))(
        states, player_per_env, features["incoming_me"], features["incoming_enemy"]
    )
    
    # Phase 2: Sample & Bucket Grid
    sampled = sample_actions(
        rng, out.target_logits, out.bucket_logits, states, phase1, deterministic=deterministic
    )
    
    # Phase 3: Pack
    actions, action_mask, executed_mask = pack_padded_actions(
        sampled["target_idx"], sampled["bucket_idx"], sampled["source_valid"],
        phase1["from_ids"], sampled["angle"], sampled["ship_counts"]
    )
    
    return {
        "actions": actions,
        "action_mask": action_mask,
        "target_idx": sampled["target_idx"],
        "bucket_idx": sampled["bucket_idx"],
        "log_prob": sampled["log_prob"],
        "entropy_target": sampled["entropy_target"],
        "entropy_bucket": sampled["entropy_bucket"],
        "executed_mask": executed_mask,
        "value": out.value,
        "target_has_bucket": phase1["target_mask"], # (B, P, P)
        "chosen_bucket_valid": sampled["chosen_bucket_valid"], # (B, P, B)
    }


In [ ]:
%%writefile src/orbit_wars/env.py
"""High-level Orbit Wars JAX environment API."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any

from .convert import state_to_observation_dict
from .reset import reset
from .state import OrbitWarsState
from .step import step


@dataclass(slots=True)
class EnvStep:
    state: OrbitWarsState
    observation: dict[str, Any]
    rewards: tuple[float, float]
    done: bool


class OrbitWarsJaxEnv:
    """Single-environment wrapper matching RL training expectations."""

    def __init__(
        self,
        *,
        seed: int = 0,
        episode_steps: int = 500,
        ship_speed: float = 6.0,
        env_root: str | None = None,
        learner_player: int = 0,
    ) -> None:
        self.seed = int(seed)
        self.episode_steps = int(episode_steps)
        self.ship_speed = float(ship_speed)
        self.env_root = env_root
        self.learner_player = int(learner_player)
        self.state: OrbitWarsState | None = None
        self._episode = 0

    def reset(self, seed: int | None = None) -> dict[str, Any]:
        if seed is not None:
            self.seed = int(seed)
        else:
            self.seed = self.seed + 9973
        self.state = reset(
            self.seed,
            episode_steps=self.episode_steps,
            ship_speed=self.ship_speed,
            env_root=self.env_root,
        )
        self._episode += 1
        return state_to_observation_dict(self.state, player=self.learner_player)

    def step(self, learner_action: list[list[float | int]], opponent_action: list[list[float | int]]) -> EnvStep:
        if self.state is None:
            raise RuntimeError("Call reset() before step().")
        if self.learner_player == 0:
            actions = [learner_action, opponent_action]
        else:
            actions = [opponent_action, learner_action]
        self.state = step(self.state, actions)
        obs = state_to_observation_dict(self.state, player=self.learner_player)
        rewards = (float(self.state.rewards[0]), float(self.state.rewards[1]))
        done = bool(self.state.done)
        return EnvStep(state=self.state, observation=obs, rewards=rewards, done=done)


class VectorOrbitWarsEnv:
    """Batched env stepping for throughput benchmarks (JIT core, no comet spawn)."""

    def __init__(self, num_envs: int, *, episode_steps: int = 500) -> None:
        self.num_envs = int(num_envs)
        self.episode_steps = int(episode_steps)
        self.states: OrbitWarsState | None = None

    def reset_batch(self, seeds: list[int]) -> list[dict[str, Any]]:
        assert len(seeds) == self.num_envs

        states = [reset(s, episode_steps=self.episode_steps) for s in seeds]
        # stack into batched struct — for benchmark use list stepping if stack fails
        self._state_list = states
        return [state_to_observation_dict(s, player=0) for s in states]

    def step_batch_noop(self) -> None:
        """Advance all envs with empty actions (benchmark helper)."""
        from .step import step

        empty: list[list[float | int]] = []
        self._state_list = [
            step(s, [empty, empty]) for s in self._state_list
        ]


In [ ]:
%%writefile src/orbit_wars/__init__.py

from .constants import *  # noqa: F403
from .convert import observation_to_state, state_to_observation_dict, states_equal
from .decode import (
    bucket_validity_mask,
    compose_target_grid,
    compose_bucket_grid,
    launch_angle,
    pack_action_row,
    path_crosses_sun,
    ship_counts_for_buckets,
)
from .env import OrbitWarsJaxEnv, VectorOrbitWarsEnv
from .features_jax import (
    ObsBatch,
    extract_obs_v8_jax,
    extract_obs_v9_jax,
)
from .reference import reference_reset, reference_step
from .reset import reset
from .state import OrbitWarsState
from .step import batched_step, step, step_jit

__all__ = [
    "OrbitWarsJaxEnv",
    "VectorOrbitWarsEnv",
    "OrbitWarsState",
    "reset",
    "step",
    "step_jit",
    "batched_step",
    "reference_reset",
    "reference_step",
    "observation_to_state",
    "state_to_observation_dict",
    "states_equal",
    "ObsBatch",
    "extract_obs_v8_jax",
    "extract_obs_v9_jax",
    "BUCKET_COUNT",
    "compose_target_grid",
    "compose_bucket_grid",
    "ship_counts_for_buckets",
    "bucket_validity_mask",
    "path_crosses_sun",
    "launch_angle",
    "pack_action_row",
]


In [ ]:
%%writefile src/orbit_wars/state.py
"""Padded Orbit Wars game state for JAX simulation."""

from __future__ import annotations

from flax import struct
import jax.numpy as jnp

from .constants import (
    DEFAULT_EPISODE_STEPS,
    DEFAULT_SHIP_SPEED,
    FLEET_COLS,
    MAX_COMET_GROUPS,
    MAX_COMET_PATH_LEN,
    MAX_COMET_PLANETS,
    MAX_FLEETS,
    MAX_PLANETS,
    NUM_PLAYERS,
    PLANET_COLS,
)


@struct.dataclass
class CometGroups:
    active: jnp.ndarray  # (MAX_COMET_GROUPS,) bool
    planet_ids: jnp.ndarray  # (MAX_COMET_GROUPS, 4) int32
    path_index: jnp.ndarray  # (MAX_COMET_GROUPS,) int32
    paths: jnp.ndarray  # (MAX_COMET_GROUPS, 4, MAX_COMET_PATH_LEN, 2) float32
    path_lengths: jnp.ndarray  # (MAX_COMET_GROUPS, 4) int32


@struct.dataclass
class OrbitWarsState:
    planets: jnp.ndarray  # (MAX_PLANETS, PLANET_COLS)
    initial_planets: jnp.ndarray  # (MAX_PLANETS, PLANET_COLS)
    n_planets: jnp.int32
    fleets: jnp.ndarray  # (MAX_FLEETS, FLEET_COLS)
    n_fleets: jnp.int32
    comets: CometGroups
    comet_planet_ids: jnp.ndarray  # (MAX_COMET_PLANETS,) int32, -1 pad
    n_comet_planet_ids: jnp.int32
    angular_velocity: jnp.float32
    step: jnp.int32
    next_fleet_id: jnp.int32
    episode_seed: jnp.int32
    done: jnp.bool_
    rewards: jnp.ndarray  # (NUM_PLAYERS,) float32
    ship_speed: jnp.float32
    episode_steps: jnp.int32


def empty_comet_groups() -> CometGroups:
    return CometGroups(
        active=jnp.zeros((MAX_COMET_GROUPS,), dtype=jnp.bool_),
        planet_ids=jnp.full((MAX_COMET_GROUPS, 4), -1, dtype=jnp.int32),
        path_index=jnp.full((MAX_COMET_GROUPS,), -1, dtype=jnp.int32),
        paths=jnp.zeros((MAX_COMET_GROUPS, 4, MAX_COMET_PATH_LEN, 2), dtype=jnp.float32),
        path_lengths=jnp.zeros((MAX_COMET_GROUPS, 4), dtype=jnp.int32),
    )


def empty_state() -> OrbitWarsState:
    return OrbitWarsState(
        planets=jnp.zeros((MAX_PLANETS, PLANET_COLS), dtype=jnp.float32),
        initial_planets=jnp.zeros((MAX_PLANETS, PLANET_COLS), dtype=jnp.float32),
        n_planets=jnp.int32(0),
        fleets=jnp.zeros((MAX_FLEETS, FLEET_COLS), dtype=jnp.float32),
        n_fleets=jnp.int32(0),
        comets=empty_comet_groups(),
        comet_planet_ids=jnp.full((MAX_COMET_PLANETS,), -1, dtype=jnp.int32),
        n_comet_planet_ids=jnp.int32(0),
        angular_velocity=jnp.float32(0.0),
        step=jnp.int32(0),
        next_fleet_id=jnp.int32(0),
        episode_seed=jnp.int32(0),
        done=jnp.bool_(False),
        rewards=jnp.zeros((NUM_PLAYERS,), dtype=jnp.float32),
        ship_speed=jnp.float32(DEFAULT_SHIP_SPEED),
        episode_steps=jnp.int32(DEFAULT_EPISODE_STEPS),
    )


In [ ]:
%%writefile src/orbit_wars/producer.py
"""JAX implementation of the 'Producer Agent' (1230 ELO) logic.
Ported from orbit_lite (PyTorch) to JAX for optimized PPO training.
"""

from __future__ import annotations

import jax
import jax.numpy as jnp
import numpy as np
from typing import NamedTuple

from .state import OrbitWarsState
from .constants import (
    CENTER, 
    SUN_RADIUS, 
    BOARD_SIZE, 
    NUM_PLAYERS,
    MAX_PLANETS,
    MAX_FLEETS,
    MAX_MOVES_PER_PLAYER,
)


# ---------------------------------------------------------------------------
# Data Types
# ---------------------------------------------------------------------------

class GarrisonStatus(NamedTuple):
    """Post-combat owner and ships over time [P, H+1]."""
    owner: jnp.ndarray
    ships: jnp.ndarray
    pre_combat_owner: jnp.ndarray
    pre_combat_ships: jnp.ndarray
    arrivals_by_owner: jnp.ndarray  # [P, H+1, A]


# ---------------------------------------------------------------------------
# Core Simulation / Prediction
# ---------------------------------------------------------------------------

def _per_step_survivor(arrivals: jnp.ndarray) -> tuple[jnp.ndarray, jnp.ndarray]:
    """Combat survivor over player axis: (owner, ships) [..., A] -> (...,)"""
    A = arrivals.shape[-1]
    if A == 2:
        s0 = arrivals[..., 0]
        s1 = arrivals[..., 1]
        top_ships = jnp.maximum(s0, s1)
        second_ships = jnp.minimum(s0, s1)
        top_owner = (s1 > s0).astype(jnp.int32)
    else:
        sorted_arr = jnp.sort(arrivals, axis=-1)
        top_ships = sorted_arr[..., -1]
        second_ships = sorted_arr[..., -2]
        top_owner = jnp.argmax(arrivals, axis=-1)
        
    tied = jnp.equal(top_ships, second_ships)
    survivor_ships = jnp.where(tied, 0.0, top_ships - second_ships)
    return top_owner, survivor_ships


def project_garrison(
    state: OrbitWarsState,
    horizon: int,
    extra_arrivals: jnp.ndarray | None = None, # [P, H, A]
) -> GarrisonStatus:
    """Project planet owner/ships over H steps using exact recurrence."""
    H = horizon
    P = state.planets.shape[0]
    A = NUM_PLAYERS
    
    init_owner = state.planets[:, 1].astype(jnp.int32)
    init_ships = state.planets[:, 5]
    prod = state.planets[:, 6]
    alive = state.planets[:, 7] > 0.0
    
    arrivals = extra_arrivals if extra_arrivals is not None else jnp.zeros((P, H, A))
    
    def step_fn(carry, t):
        curr_owner, curr_ships = carry
        produces = (curr_owner >= 0) & alive
        next_ships = curr_ships + jnp.where(produces, prod, 0.0)
        pre_owner = curr_owner
        pre_ships = next_ships
        
        s_owner, s_ships = _per_step_survivor(arrivals[:, t, :])
        has_combat = (s_ships > 0.0)
        same = (curr_owner == s_owner)
        diff = next_ships - s_ships
        attacker_wins = (~same) & (diff < 0.0)
        
        combat_ships = jnp.where(same, next_ships + s_ships, jnp.abs(diff))
        combat_owner = jnp.where(attacker_wins, s_owner, curr_owner)
        
        next_ships = jnp.where(has_combat, combat_ships, next_ships)
        next_owner = jnp.where(has_combat, combat_owner, curr_owner)
        
        next_ships = jnp.where(alive, next_ships, 0.0)
        next_owner = jnp.where(alive, next_owner, -1)
        
        return (next_owner, next_ships), (next_owner, next_ships, pre_owner, pre_ships)

    _, (o_traj, s_traj, po_traj, ps_traj) = jax.lax.scan(
        step_fn, (init_owner, init_ships), jnp.arange(H)
    )
    
    # Prepend initial state
    # Scan outputs are [H, P]. Transpose to [P, H]
    o_traj = jnp.concatenate([init_owner[:, None], o_traj.transpose(1, 0)], axis=1)
    s_traj = jnp.concatenate([init_ships[:, None], s_traj.transpose(1, 0)], axis=1)
    po_traj = jnp.concatenate([init_owner[:, None], po_traj.transpose(1, 0)], axis=1)
    ps_traj = jnp.concatenate([init_ships[:, None], ps_traj.transpose(1, 0)], axis=1)
    
    arr_full = jnp.concatenate([jnp.zeros((P, 1, A)), arrivals], axis=1)

    return GarrisonStatus(
        owner=o_traj,
        ships=s_traj,
        pre_combat_owner=po_traj,
        pre_combat_ships=ps_traj,
        arrivals_by_owner=arr_full,
    )


# ---------------------------------------------------------------------------
# Scoring / ROI
# ---------------------------------------------------------------------------

def _flow_terms_per_planet(
    status: GarrisonStatus,
    prod: jnp.ndarray, # [P]
) -> tuple[jnp.ndarray, jnp.ndarray]:
    """Calculate (produced, combat_lost) per planet per player [P, A]."""
    P, H1 = status.owner.shape
    H = H1 - 1
    A = status.arrivals_by_owner.shape[-1]
    
    owner_before = status.owner[:, :H] # [P, H]
    amount = prod[:, None] # [P, 1]
    
    a_idx = jnp.arange(A)
    prod_mask = (owner_before[:, :, None] == a_idx[None, None, :])
    produced = jnp.sum(amount[:, :, None] * prod_mask, axis=1) # [P, A]
    
    arr_k = status.arrivals_by_owner[:, 1:, :] # [P, H, A]
    s_owner, s_ships = _per_step_survivor(arr_k) # [P, H]
    
    is_survivor = (s_owner[:, :, None] == a_idx[None, None, :])
    survived_ships = jnp.where(is_survivor, s_ships[:, :, None], 0.0)
    attacker_lost = jnp.sum(jnp.maximum(0.0, arr_k - survived_ships), axis=1) # [P, A]
    
    prior_owner = status.pre_combat_owner[:, 1:] # [P, H]
    prior_ships = status.pre_combat_ships[:, 1:] # [P, H]
    fights_garrison = (s_ships > 0.0) & (s_owner != prior_owner) & (s_owner >= 0)
    g_loss = jnp.where(fights_garrison, jnp.minimum(prior_ships, s_ships), 0.0)
    
    is_prior = (prior_owner[:, :, None] == a_idx[None, None, :]) & fights_garrison[:, :, None] & (prior_owner[:, :, None] >= 0)
    is_winning_attacker = (s_owner[:, :, None] == a_idx[None, None, :]) & fights_garrison[:, :, None]
    garrison_lost = jnp.sum(g_loss[:, :, None] * (is_prior + is_winning_attacker), axis=1) # [P, A]
    
    return produced, attacker_lost + garrison_lost


def competitive_score(produced: jnp.ndarray, lost: jnp.ndarray, player_id: jnp.ndarray | int) -> jnp.ndarray:
    """ROI: (My Net Delta) - (Sum of Opponent Net Deltas)."""
    # produced, lost can be [P, A] or [A]
    net = produced - lost
    A = net.shape[-1]
    
    if net.ndim == 1:
        me = net[player_id]
        opp = jnp.sum(net) - me
        return me - opp
    else:
        # produced [P, A], player_id is scalar
        me = net[:, player_id]
        opp = jnp.sum(net, axis=-1) - me
        return me - opp


def get_heuristic_roi(state: OrbitWarsState, player_id: int, horizon: int = 20) -> jnp.ndarray:
    """Calculate the total ROI score for a player in a given state."""
    status = project_garrison(state, horizon)
    prod = state.planets[:, 6]
    p_produced, p_lost = _flow_terms_per_planet(status, prod)
    # sum over planets
    total_produced = jnp.sum(p_produced, axis=0)
    total_lost = jnp.sum(p_lost, axis=0)
    return competitive_score(total_produced, total_lost, player_id)


def safe_drain(status: GarrisonStatus, source_idx: jnp.ndarray, player_id: int) -> jnp.ndarray:
    """Calculate the minimum garrison ship count that must be preserved on source planets."""
    ships_traj = status.ships[source_idx, 1:]
    owner_traj = status.owner[source_idx, 1:]
    me_owned = (owner_traj == player_id)
    inf_fill = jnp.full_like(ships_traj, 1e9)
    cap_traj = jnp.where(me_owned & (ships_traj > 0.0), ships_traj, inf_fill)
    min_slack = jnp.min(cap_traj, axis=-1)
    current_ships = status.ships[source_idx, 0]
    return jnp.maximum(0.0, jnp.minimum(min_slack, current_ships))


In [ ]:
%%writefile src/orbit_wars/reset.py
"""Reset Orbit Wars JAX state from reference env."""

from __future__ import annotations

from .convert import observation_to_state
from .reference import episode_seed_from_env, reference_reset
from .state import OrbitWarsState


def reset(
    seed: int,
    *,
    episode_steps: int = 500,
    ship_speed: float = 6.0,
    env_root: str | None = None,
) -> OrbitWarsState:
    env, ref = reference_reset(seed, episode_steps=episode_steps, env_root=env_root)
    episode_seed = episode_seed_from_env(env)
    return observation_to_state(
        ref.observations[0],
        episode_seed=episode_seed,
        ship_speed=ship_speed,
        episode_steps=episode_steps,
        done=ref.done,
        rewards=ref.rewards,
    )


In [ ]:
%%writefile src/orbit_wars/convert.py
"""Convert between Python list observations and padded JAX state."""

from __future__ import annotations

from typing import Any

import jax.numpy as jnp
import numpy as np

from .constants import (
    FLEET_COLS,
    MAX_COMET_GROUPS,
    MAX_COMET_PATH_LEN,
    MAX_COMET_PLANETS,
    MAX_FLEETS,
    MAX_PLANETS,
    PLANET_COLS,
)
from .state import CometGroups, OrbitWarsState


def _get(obs: Any, key: str, default: Any = None) -> Any:
    if isinstance(obs, dict):
        return obs.get(key, default)
    return getattr(obs, key, default)


def _pack_planets(rows: list[list[float]], *, pad: np.ndarray) -> tuple[np.ndarray, int]:
    out = pad.copy()
    n = min(len(rows), MAX_PLANETS)
    for i, row in enumerate(rows[:n]):
        out[i, 0] = float(row[0])
        out[i, 1] = float(row[1])
        out[i, 2] = float(row[2])
        out[i, 3] = float(row[3])
        out[i, 4] = float(row[4])
        out[i, 5] = float(row[5])
        out[i, 6] = float(row[6])
        out[i, 7] = 1.0
    return out, n


def _pack_fleets(rows: list[list[float]], *, pad: np.ndarray) -> tuple[np.ndarray, int]:
    out = pad.copy()
    n = min(len(rows), MAX_FLEETS)
    for i, row in enumerate(rows[:n]):
        out[i, 0] = float(row[0])
        out[i, 1] = float(row[1])
        out[i, 2] = float(row[2])
        out[i, 3] = float(row[3])
        out[i, 4] = float(row[4])
        out[i, 5] = float(row[5])
        out[i, 6] = float(row[6])
        out[i, 7] = 1.0
    return out, n


def pack_comets(comets: list[dict[str, Any]]) -> CometGroups:
    active = np.zeros((MAX_COMET_GROUPS,), dtype=np.bool_)
    planet_ids = np.full((MAX_COMET_GROUPS, 4), -1, dtype=np.int32)
    path_index = np.full((MAX_COMET_GROUPS,), -1, dtype=np.int32)
    paths = np.zeros((MAX_COMET_GROUPS, 4, MAX_COMET_PATH_LEN, 2), dtype=np.float32)
    path_lengths = np.zeros((MAX_COMET_GROUPS, 4), dtype=np.int32)

    for gi, group in enumerate(comets[:MAX_COMET_GROUPS]):
        active[gi] = True
        path_index[gi] = int(group.get("path_index", -1))
        pids = list(group.get("planet_ids") or [])
        group_paths = list(group.get("paths") or [])
        for pi in range(min(4, len(pids))):
            planet_ids[gi, pi] = int(pids[pi])
            if pi < len(group_paths):
                path = group_paths[pi]
                plen = min(len(path), MAX_COMET_PATH_LEN)
                path_lengths[gi, pi] = plen
                for ti in range(plen):
                    paths[gi, pi, ti, 0] = float(path[ti][0])
                    paths[gi, pi, ti, 1] = float(path[ti][1])

    return CometGroups(
        active=jnp.asarray(active),
        planet_ids=jnp.asarray(planet_ids),
        path_index=jnp.asarray(path_index),
        paths=jnp.asarray(paths),
        path_lengths=jnp.asarray(path_lengths),
    )


def pack_comet_planet_ids(ids: list[int]) -> tuple[jnp.ndarray, int]:
    arr = np.full((MAX_COMET_PLANETS,), -1, dtype=np.int32)
    n = min(len(ids), MAX_COMET_PLANETS)
    for i, pid in enumerate(ids[:n]):
        arr[i] = int(pid)
    return jnp.asarray(arr), n


def observation_to_state(
    obs: Any,
    *,
    episode_seed: int = 0,
    ship_speed: float = 6.0,
    episode_steps: int = 500,
    done: bool = False,
    rewards: tuple[float, float] = (0.0, 0.0),
) -> OrbitWarsState:
    planet_pad = np.zeros((MAX_PLANETS, PLANET_COLS), dtype=np.float32)
    fleet_pad = np.zeros((MAX_FLEETS, FLEET_COLS), dtype=np.float32)

    planets, n_planets = _pack_planets(list(_get(obs, "planets", []) or []), pad=planet_pad)
    initial, _ = _pack_planets(list(_get(obs, "initial_planets", []) or []), pad=planet_pad.copy())
    fleets, n_fleets = _pack_fleets(list(_get(obs, "fleets", []) or []), pad=fleet_pad)
    comets = pack_comets(list(_get(obs, "comets", []) or []))
    comet_ids, n_comet_ids = pack_comet_planet_ids(list(_get(obs, "comet_planet_ids", []) or []))

    return OrbitWarsState(
        planets=jnp.asarray(planets),
        initial_planets=jnp.asarray(initial),
        n_planets=jnp.int32(n_planets),
        fleets=jnp.asarray(fleets),
        n_fleets=jnp.int32(n_fleets),
        comets=comets,
        comet_planet_ids=comet_ids,
        n_comet_planet_ids=jnp.int32(n_comet_ids),
        angular_velocity=jnp.float32(float(_get(obs, "angular_velocity", 0.0))),
        step=jnp.int32(int(_get(obs, "step", 0))),
        next_fleet_id=jnp.int32(int(_get(obs, "next_fleet_id", 0))),
        episode_seed=jnp.int32(int(episode_seed)),
        done=jnp.bool_(done),
        rewards=jnp.asarray(rewards, dtype=jnp.float32),
        ship_speed=jnp.float32(float(ship_speed)),
        episode_steps=jnp.int32(int(episode_steps)),
    )


def _planet_rows(state: OrbitWarsState) -> list[list[float]]:
    rows: list[list[float]] = []
    planets = np.asarray(state.planets)
    for i in range(int(state.n_planets)):
        if planets[i, 7] <= 0.0:
            continue
        rows.append(
            [
                float(planets[i, 0]),
                float(planets[i, 1]),
                float(planets[i, 2]),
                float(planets[i, 3]),
                float(planets[i, 4]),
                float(planets[i, 5]),
                float(planets[i, 6]),
            ]
        )
    return rows


def _fleet_rows(state: OrbitWarsState) -> list[list[float]]:
    rows: list[list[float]] = []
    fleets = np.asarray(state.fleets)
    for i in range(int(state.n_fleets)):
        if fleets[i, 7] <= 0.0:
            continue
        rows.append(
            [
                float(fleets[i, 0]),
                float(fleets[i, 1]),
                float(fleets[i, 2]),
                float(fleets[i, 3]),
                float(fleets[i, 4]),
                float(fleets[i, 5]),
                float(fleets[i, 6]),
            ]
        )
    return rows


def _unpack_comets(comets: CometGroups) -> list[dict[str, Any]]:
    active = np.asarray(comets.active)
    planet_ids = np.asarray(comets.planet_ids)
    path_index = np.asarray(comets.path_index)
    paths = np.asarray(comets.paths)
    path_lengths = np.asarray(comets.path_lengths)
    out: list[dict[str, Any]] = []
    for gi in range(MAX_COMET_GROUPS):
        if not active[gi]:
            continue
        group_paths: list[list[list[float]]] = []
        pids: list[int] = []
        for pi in range(4):
            pid = int(planet_ids[gi, pi])
            if pid < 0:
                continue
            pids.append(pid)
            plen = int(path_lengths[gi, pi])
            group_paths.append(
                [[float(paths[gi, pi, ti, 0]), float(paths[gi, pi, ti, 1])] for ti in range(plen)]
            )
        out.append({"planet_ids": pids, "paths": group_paths, "path_index": int(path_index[gi])})
    return out


def state_to_observation_dict(state: OrbitWarsState, *, player: int = 0) -> dict[str, Any]:
    comet_ids = [int(x) for x in np.asarray(state.comet_planet_ids)[: int(state.n_comet_planet_ids)] if int(x) >= 0]
    return {
        "step": int(state.step),
        "player": int(player),
        "planets": _planet_rows(state),
        "initial_planets": _planet_rows(
            OrbitWarsState(
                planets=state.initial_planets,
                initial_planets=state.initial_planets,
                n_planets=state.n_planets,
                fleets=state.fleets,
                n_fleets=state.n_fleets,
                comets=state.comets,
                comet_planet_ids=state.comet_planet_ids,
                n_comet_planet_ids=state.n_comet_planet_ids,
                angular_velocity=state.angular_velocity,
                step=state.step,
                next_fleet_id=state.next_fleet_id,
                episode_seed=state.episode_seed,
                done=state.done,
                rewards=state.rewards,
                ship_speed=state.ship_speed,
                episode_steps=state.episode_steps,
            )
        ),
        "fleets": _fleet_rows(state),
        "angular_velocity": float(state.angular_velocity),
        "next_fleet_id": int(state.next_fleet_id),
        "comets": _unpack_comets(state.comets),
        "comet_planet_ids": comet_ids,
    }


def states_equal(a: OrbitWarsState, b: OrbitWarsState, *, atol: float = 1e-4) -> bool:
    """Compare simulation-relevant fields (ignores padded inactive slots)."""
    if int(a.n_planets) != int(b.n_planets) or int(a.n_fleets) != int(b.n_fleets):
        return False
    if int(a.step) != int(b.step) or bool(a.done) != bool(b.done):
        return False
    ap = np.asarray(a.planets)[: int(a.n_planets), :7]
    bp = np.asarray(b.planets)[: int(b.n_planets), :7]
    af = np.asarray(a.fleets)[: int(a.n_fleets), :7]
    bf = np.asarray(b.fleets)[: int(b.n_fleets), :7]
    if not np.allclose(ap, bp, atol=atol, rtol=0.0):
        return False
    if not np.allclose(af, bf, atol=atol, rtol=0.0):
        return False
    return True


In [ ]:
%%writefile src/orbit_wars/features_jax.py
import jax
import jax.numpy as jnp
from jaxtyping import Array, Float, Int32, Bool
from env.jax_orbit_wars import JaxEnvState, plan_target_shot_jax, fleet_speed_jax, CENTER, SUN_RADIUS

MAX_SHIPS = 400.0
MAX_PRODUCTION = 5.0
MIN_LAUNCH_SHIPS = 5.0
FUTURE_ORACLE_STEPS = 32
FUTURE_ORACLE_SCALE = 30.0
N_SHIP_OPTIONS = 3


@jax.jit
def ship_options_for_edge_jax(src_ships: Array, tgt_ships: Array, tgt_owner: Array, player_id: Array) -> Array:
    """Returns the 3 source-fraction bins: 50%, 75%, and 100% of source ships."""
    pct_50 = jnp.maximum(1.0, jnp.round(0.50 * src_ships))
    pct_75 = jnp.maximum(1.0, jnp.round(0.75 * src_ships))
    pct_100 = src_ships

    return jnp.stack([pct_50, pct_75, pct_100], axis=-1)


@jax.jit
def extract_node_features_jax(state: JaxEnvState, player_id: Array) -> Array:
    """Extracts 12 player-index-invariant node features for all 60 planets."""
    cur_turn = state.cur_turn
    owners = state.future_timeline[:, cur_turn, 0]
    ships = state.future_timeline[:, cur_turn, 1]
    positions = state.planet_positions_all_turns[:, cur_turn]
    
    is_owned_by_player = (owners == player_id).astype(jnp.float32)
    is_owned_by_opponent = ((owners != player_id) & (owners != -1)).astype(jnp.float32)
    is_neutral = (owners == -1).astype(jnp.float32)
    
    x_norm = positions[:, 0] / 100.0
    y_norm = positions[:, 1] / 100.0
    r_norm = jnp.minimum(state.planet_radius / 10.0, 1.0)
    s_norm = jnp.minimum(ships / MAX_SHIPS, 1.0)
    p_norm = jnp.minimum(state.planet_production / MAX_PRODUCTION, 1.0)
    
    is_rot = state.is_rotating.astype(jnp.float32)
    is_com = state.is_comet.astype(jnp.float32)
    
    dist_to_sun = jnp.sqrt(jnp.sum((positions - CENTER) ** 2, axis=1)) / 70.71
    
    # Dynamic Fleet Pressure from precomputed future incoming fleets
    inc = state.incoming_fleets
    
    # Mask out past turns
    turns_mask = (jnp.arange(500) > cur_turn)[None, :, None]
    valid_inc = jnp.where(turns_mask, inc, 0.0)
    
    # Sum over all opponents
    opp_arrivals = jnp.sum(valid_inc, axis=-1) - valid_inc[:, :, player_id]
    net_incoming = jnp.sum(opp_arrivals - valid_inc[:, :, player_id], axis=1)
    fleet_pressure = jnp.clip(net_incoming / MAX_SHIPS, -1.0, 1.0)
    
    feats = jnp.stack([
        is_owned_by_player,
        is_owned_by_opponent,
        is_neutral,
        x_norm,
        y_norm,
        r_norm,
        s_norm,
        p_norm,
        is_rot,
        is_com,
        dist_to_sun,
        fleet_pressure
    ], axis=-1)
    
    # Apply active mask to zero out inactive slots, except neutral/dist_to_sun
    feats = feats * state.active_mask[:, None]
    feats = feats.at[:, 2].set(jnp.where(state.active_mask, feats[:, 2], 1.0))
    feats = feats.at[:, 10].set(jnp.where(state.active_mask, feats[:, 10], 1.0))
    
    return feats


@jax.jit
def extract_node_features_v8_jax(state: JaxEnvState, player_id: Array) -> Array:
    """Extracts V8's 21 node features: 12 base features plus 3 facts for each ship bucket."""
    base = extract_node_features_jax(state, player_id)
    cur_turn = state.cur_turn
    ships = state.future_timeline[:, cur_turn, 1]

    def bucket_block(pct):
        bucket_ships = jnp.maximum(1.0, jnp.floor(pct * ships + 0.5))
        speed = fleet_speed_jax(bucket_ships)
        return jnp.stack(
            [
                jnp.minimum(bucket_ships / MAX_SHIPS, 1.0),
                speed / 6.0,
                (bucket_ships >= 20.0).astype(jnp.float32),
            ],
            axis=-1,
        )

    buckets = jnp.concatenate(
        [
            bucket_block(0.50),
            bucket_block(0.75),
            bucket_block(1.00),
        ],
        axis=-1,
    )
    buckets = buckets * state.active_mask[:, None]
    return jnp.concatenate([base, buckets], axis=-1)


@jax.jit
def extract_future_sight_jax(state: JaxEnvState, player_id: Array) -> Array:
    """Extracts sliding-window 32-step future timelines in JAX."""
    cur_turn = state.cur_turn
    timeline = state.future_timeline  # (60, 500, 2)
    
    # 32-step slicing
    t_indices = jnp.clip(cur_turn + 1 + jnp.arange(FUTURE_ORACLE_STEPS), 0, 499)
    
    # Sliced owners and ships: (60, 32)
    sliced_owners = timeline[:, t_indices, 0]
    sliced_ships = timeline[:, t_indices, 1]
    
    is_player = sliced_owners == player_id
    is_opponent = (sliced_owners != player_id) & (sliced_owners != -1)
    
    val = jnp.where(is_player, sliced_ships, jnp.where(is_opponent, -sliced_ships, 0.0))
    future_sight = val / FUTURE_ORACLE_SCALE
    future_sight = future_sight * state.active_mask[:, None]
    
    return future_sight


@jax.jit
def extract_edge_features_jax(state: JaxEnvState, player_id: Array) -> tuple[Array, Array, Array]:
    """Generates all-to-all edge index and 9 edge features in JAX."""
    cur_turn = state.cur_turn
    positions = state.planet_positions_all_turns[:, cur_turn]
    ships = state.future_timeline[:, cur_turn, 1]
    owners = state.future_timeline[:, cur_turn, 0]
    
    # Create all pair combinations
    p_count = 60
    src, dst = jnp.meshgrid(jnp.arange(p_count), jnp.arange(p_count), indexing="ij")
    src = src.reshape(-1)
    dst = dst.reshape(-1)
    
    # Edge index: (2, 3600)
    edge_index = jnp.stack([src, dst], axis=0)
    
    diff = positions[dst] - positions[src]
    dist = jnp.sqrt(jnp.sum(diff * diff, axis=-1))
    direct_angle = jnp.arctan2(diff[:, 1], diff[:, 0])
    
    # Ship options and speeds
    src_ships = ships[src]
    tgt_ships = ships[dst]
    tgt_owners = owners[dst]
    
    options = ship_options_for_edge_jax(src_ships, tgt_ships, tgt_owners, player_id)
    ref_ships = options[:, 0]  # 50% source bucket as static reference ship count
    
    speed = fleet_speed_jax(ref_ships)
    
    sin_a = jnp.sin(direct_angle)
    cos_a = jnp.cos(direct_angle)
    launch_clearance = state.planet_radius[src] + 0.1
    start_x = positions[src, 0] + cos_a * launch_clearance
    start_y = positions[src, 1] + sin_a * launch_clearance
    
    to_target_x = positions[dst, 0] - start_x
    to_target_y = positions[dst, 1] - start_y
    launch_dist = jnp.sqrt(to_target_x * to_target_x + to_target_y * to_target_y)
    turns = launch_dist / jnp.maximum(speed, 1e-6)
    
    # Check sun intersection
    seg_x = positions[dst, 0] - start_x
    seg_y = positions[dst, 1] - start_y
    seg_len_sq = jnp.maximum(seg_x * seg_x + seg_y * seg_y, 1e-9)
    proj = ((CENTER - start_x) * seg_x + (CENTER - start_y) * seg_y) / seg_len_sq
    proj = jnp.clip(proj, 0.0, 1.0)
    close_x = start_x + proj * seg_x
    close_y = start_y + proj * seg_y
    sun_dist = jnp.sqrt((CENTER - close_x) ** 2 + (CENTER - close_y) ** 2)
    crosses_sun = (sun_dist < SUN_RADIUS).astype(jnp.float32)
    
    # Construct Edge Features: (3600, 9)
    feat0 = diff[:, 0] / 100.0
    feat1 = diff[:, 1] / 100.0
    feat2 = dist / (100.0 * jnp.sqrt(2.0))
    feat3 = sin_a
    feat4 = cos_a
    feat5 = jnp.minimum(ref_ships / MAX_SHIPS, 1.0)
    
    desired_ships = jnp.maximum(tgt_ships + 1.0, 20.0)
    feat6 = (src_ships >= desired_ships).astype(jnp.float32)
    feat7 = crosses_sun
    feat8 = jnp.minimum(turns / 100.0, 1.0)
    
    edge_features = jnp.stack([
        feat0, feat1, feat2, feat3, feat4, feat5, feat6, feat7, feat8
    ], axis=-1)
    
    return edge_index, edge_features, options


@jax.jit
def extract_edge_features_v8_jax(state: JaxEnvState, player_id: Array) -> Array:
    """Generate V8's 14 all-to-all edge features."""
    cur_turn = state.cur_turn
    _edge_index, base_flat, _ = extract_edge_features_jax(state, player_id)
    base = base_flat.reshape(60, 60, 9)

    positions = state.planet_positions_all_turns[:, cur_turn]
    ships = state.future_timeline[:, cur_turn, 1]
    diff = positions[None, :, :] - positions[:, None, :]
    direct = jnp.arctan2(diff[..., 1], diff[..., 0])
    sin_a = jnp.sin(direct)
    cos_a = jnp.cos(direct)
    start_x = positions[:, None, 0] + cos_a * (state.planet_radius[:, None] + 0.1)
    start_y = positions[:, None, 1] + sin_a * (state.planet_radius[:, None] + 0.1)
    seg_x = positions[None, :, 0] - start_x
    seg_y = positions[None, :, 1] - start_y
    rough_dist = jnp.sqrt(seg_x * seg_x + seg_y * seg_y + 1e-9)

    src_ships = ships[:, None]
    tgt_ships = ships[None, :]

    def bucket_edge_block(pct):
        bucket_ships = jnp.maximum(1.0, jnp.floor(pct * src_ships + 0.5))
        speed = fleet_speed_jax(bucket_ships)
        rough_turns = jnp.minimum(rough_dist / jnp.maximum(speed, 1e-6) / 100.0, 1.0)
        can_clear = (bucket_ships >= (tgt_ships + 1.0)).astype(jnp.float32)
        return rough_turns, can_clear

    turns50, clear50 = bucket_edge_block(0.50)
    turns75, clear75 = bucket_edge_block(0.75)
    turns100, clear100 = bucket_edge_block(1.00)
    ratio = jnp.clip(src_ships / jnp.maximum(tgt_ships, 1.0), 0.0, 20.0) / 20.0
    roi = jnp.zeros((60, 60), dtype=jnp.float32)

    geometry = base[..., jnp.array([0, 1, 2, 3, 4, 7])]
    edges = jnp.concatenate(
        [
            geometry,
            roi[..., None],
            ratio[..., None],
            turns50[..., None],
            turns75[..., None],
            turns100[..., None],
            clear50[..., None],
            clear75[..., None],
            clear100[..., None],
        ],
        axis=-1,
    )
    edge_mask = (state.active_mask[:, None] & state.active_mask[None, :])[:, :, None]
    return edges * edge_mask


@jax.jit
def extract_owned_nodes_jax(state: JaxEnvState, player_id: Array) -> Array:
    """Finds up to 60 active planet indices owned by the player, padded with -1."""
    cur_turn = state.cur_turn
    owners = state.future_timeline[:, cur_turn, 0]
    is_active = state.active_mask
    
    is_owned = (owners == player_id) & is_active
    owned_indices = jnp.where(is_owned, jnp.arange(60), 999)
    
    # Sort to bring owned nodes to the front
    sorted_owned = jnp.sort(owned_indices)
    
    # Replace padding with -1
    final_owned = jnp.where(sorted_owned < 60, sorted_owned, -1)

    return final_owned[:60]


@jax.jit
def compute_edge_valid_mask_jax(
    state: JaxEnvState,
    owned_nodes: Array,
    player_id: Array,
) -> Array:
    """Construct the (60, 60, N_SHIP_OPTIONS) action mask from trajectory legality."""
    cur_turn = state.cur_turn
    positions = state.planet_positions_all_turns[:, cur_turn]  # (60, 2)
    ids = jnp.arange(60)
    sun_center = jnp.array([CENTER, CENTER])

    def check_slot(slot):
        src = owned_nodes[slot]
        src_safe = jnp.where(src >= 0, src, 0)
        
        src_ships = state.future_timeline[src_safe, cur_turn, 1]
        src_valid = src >= 0
        src_pos = positions[src_safe]
        seg = positions - src_pos[None, :]  # (60, 2)
        seg_len_sq = jnp.maximum(jnp.sum(seg * seg, axis=-1), 1e-9)  # (60,)

        to_sun = sun_center - src_pos
        sun_proj = jnp.clip(jnp.sum(to_sun[None, :] * seg, axis=-1) / seg_len_sq, 0.0, 1.0)
        sun_closest = src_pos[None, :] + sun_proj[:, None] * seg
        sun_dist = jnp.sqrt(jnp.sum((sun_center[None, :] - sun_closest) ** 2, axis=-1))
        blocks_sun = sun_dist < SUN_RADIUS

        blocker_vec = positions[None, :, :] - src_pos[None, None, :]  # (1, 60, 2)
        seg_t = seg[:, None, :]  # (60, 1, 2)
        proj = jnp.sum(blocker_vec * seg_t, axis=-1) / seg_len_sq[:, None]  # (60, 60)
        closest = src_pos[None, None, :] + proj[:, :, None] * seg_t
        blocker_dist = jnp.sqrt(jnp.sum((positions[None, :, :] - closest) ** 2, axis=-1))
        not_endpoint = (ids[None, :] != src_safe) & (ids[None, :] != ids[:, None])
        blocks_planet = (
            state.active_mask[None, :]
            & not_endpoint
            & (proj > 0.0)
            & (proj < 1.0)
            & (blocker_dist <= state.planet_radius[None, :])
        )
        path_blocked = blocks_sun | jnp.any(blocks_planet, axis=-1)
        
        def check_target(tgt):
            tgt_active = state.active_mask[tgt]
            is_self = src == tgt
            not_self = ~is_self
            
            base_valid = src_valid & tgt_active & not_self & (~path_blocked[tgt])
            
            opts = ship_options_for_edge_jax(src_ships, tgt_ships=state.future_timeline[tgt, cur_turn, 1], tgt_owner=state.future_timeline[tgt, cur_turn, 0], player_id=player_id)
            
            def check_opt(opt):
                ships = opts[opt]
                ships_valid = (ships >= MIN_LAUNCH_SHIPS) & (ships <= src_ships)
                noop_valid = src_valid & is_self & (opt == 0)
                return noop_valid | (base_valid & ships_valid)
            
            return jax.vmap(check_opt)(jnp.arange(N_SHIP_OPTIONS))
        
        return jax.vmap(check_target)(jnp.arange(60))
    
    mask = jax.vmap(check_slot)(jnp.arange(60))  # (60, 60, N_SHIP_OPTIONS)
    return mask


@jax.jit
def compute_edge_valid_mask_raytrace_jax(
    state: JaxEnvState,
    owned_nodes: Array,
    player_id: Array,
) -> Array:
    """Constructs the (60, 60, N_SHIP_OPTIONS) exact target-shot validity mask."""
    cur_turn = state.cur_turn
    
    def check_validity(slot, tgt, opt):
        src = owned_nodes[slot]
        src_valid = src >= 0
        is_self = src == tgt
        tgt_valid = state.active_mask[tgt] & (~is_self)
        base_valid = src_valid & tgt_valid
        
        src_ships = state.future_timeline[src, cur_turn, 1]
        tgt_ships = state.future_timeline[tgt, cur_turn, 1]
        tgt_owner = state.future_timeline[tgt, cur_turn, 0]
        
        opts = ship_options_for_edge_jax(src_ships, tgt_ships, tgt_owner, player_id)
        ships = opts[opt]
        ships_valid = (ships >= MIN_LAUNCH_SHIPS) & (ships <= src_ships)
        noop_valid = src_valid & is_self & (opt == 0)
        valid = noop_valid | (base_valid & ships_valid)
        
        _, _, _, planned_viable = plan_target_shot_jax(
            state,
            src,
            tgt,
            ships,
            cur_turn,
        )
        
        return noop_valid | (valid & planned_viable)

    mask = jax.vmap(
        lambda slot: jax.vmap(
            lambda tgt: jax.vmap(
                lambda opt: check_validity(slot, tgt, opt)
            )(jnp.arange(N_SHIP_OPTIONS))
        )(jnp.arange(60))
    )(jnp.arange(60))
    
    return mask


@jax.jit
def extract_global_features_jax(state: JaxEnvState, player_id: Array) -> Array:
    """Extracts 8 global game state features for the player."""
    cur_turn = state.cur_turn
    
    owners = state.future_timeline[:, cur_turn, 0]
    ships = state.future_timeline[:, cur_turn, 1]
    active = state.active_mask
    
    my_planets = (owners == player_id) & active
    enemy_planets = (owners != player_id) & (owners != -1) & active
    neutral_planets = (owners == -1) & active
    
    MAX_P = 60.0
    MAX_S = 400.0
    denom = MAX_P * MAX_S
    
    # Approximate fleet totals from incoming_fleets (future arrivals)
    turns_mask = (jnp.arange(500) > cur_turn)[None, :, None]
    valid_inc = jnp.where(turns_mask, state.incoming_fleets, 0.0)
    my_fleet_ships = jnp.sum(valid_inc[:, :, player_id])
    enemy_fleet_ships = jnp.sum(valid_inc) - my_fleet_ships
    
    return jnp.stack([
        jnp.minimum(cur_turn / 500.0, 1.0),
        my_planets.sum() / MAX_P,
        enemy_planets.sum() / MAX_P,
        neutral_planets.sum() / MAX_P,
        jnp.sum(jnp.where(my_planets, ships, 0.0)) / denom,
        jnp.sum(jnp.where(enemy_planets, ships, 0.0)) / denom,
        jnp.minimum(my_fleet_ships / denom, 1.0),
        jnp.minimum(enemy_fleet_ships / denom, 1.0),
    ])


@jax.jit
def state_to_graph_jax(state: JaxEnvState, player_id: Array) -> tuple[
    Array,  # node_features (60, 12)
    Array,  # edge_index (2, 3600)
    Array,  # edge_features (3600, 9)
    Array,  # future_sight (60, 32)
    Array,  # global_features (8,)
    Array,  # owned_nodes (60,)
    Array,  # edge_valid_mask (60, 60, N_SHIP_OPTIONS) - simplified
]:
    """Top-level JAX-compiled features extractor."""
    node_features = extract_node_features_jax(state, player_id)
    edge_index, edge_features, _ = extract_edge_features_jax(state, player_id)
    future_sight = extract_future_sight_jax(state, player_id)
    global_features = extract_global_features_jax(state, player_id)
    owned_nodes = extract_owned_nodes_jax(state, player_id)
    edge_valid_mask = compute_edge_valid_mask_jax(state, owned_nodes, player_id)
    
    return node_features, edge_index, edge_features, future_sight, global_features, owned_nodes, edge_valid_mask


from typing import NamedTuple

class ObsBatch(NamedTuple):
    node_features: jnp.ndarray    # (60, 21)
    edge_features: jnp.ndarray    # (60, 60, 14)
    future_sight: jnp.ndarray     # (60, 32)
    global_features: jnp.ndarray  # (8,)
    owned_nodes: jnp.ndarray      # (60,) padded with -1
    edge_valid_mask: jnp.ndarray  # (60, 60, 3)


@jax.jit
def extract_obs_v8_jax(state: JaxEnvState, player_id: Array) -> ObsBatch:
    node_features = extract_node_features_v8_jax(state, player_id)
    edge_features = extract_edge_features_v8_jax(state, player_id)
    future_sight = extract_future_sight_jax(state, player_id)
    global_features = extract_global_features_jax(state, player_id)
    owned_nodes = extract_owned_nodes_jax(state, player_id)
    edge_valid_mask = compute_edge_valid_mask_jax(state, owned_nodes, player_id)
    
    return ObsBatch(
        node_features=node_features,
        edge_features=edge_features,
        future_sight=future_sight,
        global_features=global_features,
        owned_nodes=owned_nodes,
        edge_valid_mask=edge_valid_mask,
    )


@jax.jit
def extract_obs_v9_jax(state: JaxEnvState, player_id: Array) -> ObsBatch:
    obs = extract_obs_v8_jax(state, player_id)
    return ObsBatch(
        node_features=obs.node_features,
        edge_features=obs.edge_features.astype(jnp.bfloat16),
        future_sight=obs.future_sight.astype(jnp.bfloat16),
        global_features=obs.global_features,
        owned_nodes=obs.owned_nodes,
        edge_valid_mask=obs.edge_valid_mask,
    )


In [ ]:
%%writefile src/orbit_wars/geometry.py
"""Pure JAX geometry helpers for Orbit Wars fleet/planet collision.
Updated with logic from the 1100 ELO heuristic notebook.
"""

from __future__ import annotations

import jax
import jax.numpy as jnp

from .constants import BOARD_SIZE, CENTER, SUN_RADIUS


def _match_rank(arr: jnp.ndarray, ref: jnp.ndarray) -> jnp.ndarray:
    """Expand arr with trailing 1-dims to match ref's rank."""
    while arr.ndim < ref.ndim:
        arr = arr[..., None]
    return arr


def distance_xy(x1: jnp.ndarray, y1: jnp.ndarray, x2: jnp.ndarray, y2: jnp.ndarray) -> jnp.ndarray:
    dx = x1 - x2
    dy = y1 - y2
    return jnp.sqrt(dx * dx + dy * dy)


def in_bounds(x: jnp.ndarray, y: jnp.ndarray) -> jnp.ndarray:
    return (x >= 0) & (x <= BOARD_SIZE) & (y >= 0) & (y <= BOARD_SIZE)


def point_to_segment_distance(
    px: jnp.ndarray,
    py: jnp.ndarray,
    x1: jnp.ndarray,
    y1: jnp.ndarray,
    x2: jnp.ndarray,
    y2: jnp.ndarray,
) -> jnp.ndarray:
    """Distance from point(s) (px, py) to line segment(s) (x1, y1) -> (x2, y2)."""
    dx = x2 - x1
    dy = y2 - y1
    d2 = dx * dx + dy * dy

    # Projection of point onto line (normalized to [0, 1])
    t = ((px - x1) * dx + (py - y1) * dy) / jnp.maximum(d2, 1e-12)
    t = jnp.clip(t, 0.0, 1.0)

    # Closest point on segment
    closest_x = x1 + t * dx
    closest_y = y1 + t * dy

    return distance_xy(px, py, closest_x, closest_y)


def sun_hit(
    x1: jnp.ndarray, y1: jnp.ndarray, x2: jnp.ndarray, y2: jnp.ndarray, margin: float = 1.0
) -> jnp.ndarray:
    """Does the path from (x1, y1) to (x2, y2) hit the sun?"""
    d = point_to_segment_distance(
        jnp.float32(CENTER), jnp.float32(CENTER), x1, y1, x2, y2
    )
    return d <= (SUN_RADIUS + margin)


def is_orbiting_planet(x: jnp.ndarray, y: jnp.ndarray, r: jnp.ndarray) -> jnp.ndarray:
    dx = x - CENTER
    dy = y - CENTER
    d = jnp.sqrt(dx * dx + dy * dy)
    # Match the threshold used in the official env (ROTATION_RADIUS_LIMIT = 50)
    return d + r < 50.0


def predict_orbit_polar(
    x: jnp.ndarray,
    y: jnp.ndarray,
    angular_velocity: jnp.ndarray,
    turns_ahead: jnp.ndarray,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    dx = x - CENTER
    dy = y - CENTER
    theta = jnp.arctan2(dy, dx)
    orbit_r = jnp.sqrt(dx * dx + dy * dy)
    
    # Broadcast to match turns_ahead rank
    theta = _match_rank(theta, turns_ahead)
    orbit_r = _match_rank(orbit_r, turns_ahead)
    omega = _match_rank(angular_velocity, turns_ahead)

    theta2 = theta + omega * turns_ahead
    new_x = CENTER + orbit_r * jnp.cos(theta2)
    new_y = CENTER + orbit_r * jnp.sin(theta2)
    return new_x, new_y


def predict_planet_position(
    x: jnp.ndarray,
    y: jnp.ndarray,
    is_orbiting: jnp.ndarray,
    turns_ahead: jnp.ndarray,
    angular_velocity: jnp.ndarray,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    px, py = predict_orbit_polar(x, y, angular_velocity, turns_ahead)
    # Broadcast is_orbiting to match
    orb = _match_rank(is_orbiting, turns_ahead)
    xx = _match_rank(x, turns_ahead)
    yy = _match_rank(y, turns_ahead)
    return jnp.where(orb, px, xx), jnp.where(orb, py, yy)


def precompute_comet_trajectories(
    comet_active: jnp.ndarray,
    comet_pids: jnp.ndarray,
    comet_path_index: jnp.ndarray,
    comet_paths: jnp.ndarray,
    comet_path_lengths: jnp.ndarray,
    planet_ids: jnp.ndarray,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    MAX_LEN = comet_paths.shape[2]
    G = comet_active.shape[0]
    
    flat_active = jnp.repeat(comet_active, 4)
    flat_pids = comet_pids.reshape(-1)
    flat_paths = comet_paths.reshape(-1, MAX_LEN, 2)
    flat_plens = comet_path_lengths.reshape(-1)
    flat_group_idx = jnp.repeat(jnp.arange(G), 4)

    match_all = (flat_pids[None, :] == planet_ids[:, None]) & flat_active[None, :] & (planet_ids[:, None] >= 0)
    best_slot = jnp.argmax(match_all.astype(jnp.int32), axis=-1)  # (P,)
    is_comet = jnp.any(match_all, axis=-1)  # (P,)
    
    g_idx = jnp.take(flat_group_idx, best_slot)
    p_idx = best_slot 
    
    c_path_idx = jnp.take(comet_path_index, g_idx)  # (P,)
    
    t_range = jnp.arange(MAX_LEN)[None, :]  # (1, L)
    future_idx = c_path_idx[:, None] + t_range  # (P, L)
    safe_future_idx = jnp.clip(future_idx, 0, MAX_LEN - 1)
    
    trajectories = flat_paths[p_idx[:, None], safe_future_idx]  # (P, L, 2)
    plen = flat_plens[p_idx]  # (P,)
    valid_time = (future_idx < plen[:, None]) & (future_idx >= 0)  # (P, L)
    
    return is_comet, trajectories, valid_time


def predict_target_position_fast(
    tgt_x: jnp.ndarray,
    tgt_y: jnp.ndarray,
    tgt_is_orbiting: jnp.ndarray,
    tgt_is_comet: jnp.ndarray,
    tgt_traj: jnp.ndarray,
    tgt_valid_time: jnp.ndarray,
    turns_ahead: jnp.ndarray,
    angular_velocity: jnp.ndarray,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    MAX_LEN = tgt_traj.shape[-2]
    safe_idx = jnp.clip(turns_ahead.astype(jnp.int32), 0, MAX_LEN - 1)
    
    orig_shape = safe_idx.shape
    flat_idx = safe_idx.reshape(-1)
    flat_traj = tgt_traj.reshape(-1, MAX_LEN, 2)
    
    res = flat_traj[jnp.arange(flat_idx.shape[0]), flat_idx]
    cx = res[:, 0].reshape(orig_shape)
    cy = res[:, 1].reshape(orig_shape)
    
    # 2. Orbital path
    ox, oy = predict_planet_position(tgt_x, tgt_y, tgt_is_orbiting, turns_ahead, angular_velocity)
    
    is_com = _match_rank(tgt_is_comet, turns_ahead)
    return jnp.where(is_com, cx, ox), jnp.where(is_com, cy, oy)


def get_arrival_turns(
    sx: jnp.ndarray, sy: jnp.ndarray, sr: jnp.ndarray,
    tx: jnp.ndarray, ty: jnp.ndarray, tr: jnp.ndarray,
    ships: jnp.ndarray, max_speed: jnp.ndarray,
) -> jnp.ndarray:
    d = distance_xy(sx, sy, tx, ty)
    tr = _match_rank(tr, d)
    sr = _match_rank(sr, d)
    
    hit_d = jnp.maximum(0.0, d - (sr + 0.1) - tr)
    speed = fleet_speed(ships, max_speed)
    
    hit_d_b = _match_rank(hit_d, speed)
    return jnp.maximum(1.0, jnp.ceil(hit_d_b / jnp.maximum(speed, 1e-6)))


def solve_intercept_with_wait(
    src_x: jnp.ndarray,
    src_y: jnp.ndarray,
    src_r: jnp.ndarray,
    tgt_x: jnp.ndarray,
    tgt_y: jnp.ndarray,
    tgt_r: jnp.ndarray,
    tgt_is_orbiting: jnp.ndarray,
    tgt_is_comet: jnp.ndarray,
    tgt_traj: jnp.ndarray,
    tgt_valid_time: jnp.ndarray,
    ship_count: jnp.ndarray,
    angular_velocity: jnp.ndarray,
    max_speed: jnp.ndarray,
    sun_margin: float = 1.5,
    n_iter: int = 6,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    sx = jnp.asarray(src_x).astype(jnp.float32)
    sy = jnp.asarray(src_y).astype(jnp.float32)
    sr = jnp.asarray(src_r).astype(jnp.float32)
    tx = jnp.asarray(tgt_x).astype(jnp.float32)
    ty = jnp.asarray(tgt_y).astype(jnp.float32)
    tr = jnp.asarray(tgt_r).astype(jnp.float32)
    is_orb = jnp.asarray(tgt_is_orbiting).astype(jnp.bool_)
    is_com = jnp.asarray(tgt_is_comet).astype(jnp.bool_)
    count = jnp.asarray(ship_count).astype(jnp.float32)
    speed = jnp.asarray(max_speed).astype(jnp.float32)
    
    turns = get_arrival_turns(sx, sy, sr, tx, ty, tr, count, speed)

    def body(_i, carry):
        tt, _ix, _iy = carry
        ix, iy = predict_target_position_fast(tx, ty, is_orb, is_com, tgt_traj, tgt_valid_time, tt, angular_velocity)
        tt_new = get_arrival_turns(sx, sy, sr, ix, iy, tr, count, speed)
        return tt_new, ix, iy

    ix0, iy0 = predict_target_position_fast(tx, ty, is_orb, is_com, tgt_traj, tgt_valid_time, turns, angular_velocity)
    turns, aim_x, aim_y = jax.lax.fori_loop(0, n_iter, body, (turns, ix0, iy0))
    
    sx_b = _match_rank(sx, aim_x)
    sy_b = _match_rank(sy, aim_y)
    blocked = sun_hit(sx_b, sy_b, aim_x, aim_y, margin=sun_margin)

    return aim_x, aim_y, turns, blocked


def solve_intercept(
    src_x: jnp.ndarray,
    src_y: jnp.ndarray,
    tgt_x: jnp.ndarray,
    tgt_y: jnp.ndarray,
    tgt_is_orbiting: jnp.ndarray,
    ship_count: jnp.ndarray,
    angular_velocity: jnp.ndarray,
    max_speed: jnp.ndarray,
    n_iter: int = 25,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    aim_x, aim_y, turns, _blocked = solve_intercept_with_wait(
        src_x, src_y, 0.0, 
        tgt_x, tgt_y, 0.0, tgt_is_orbiting, jnp.zeros_like(tgt_is_orbiting),
        jnp.zeros((tgt_x.shape + (1, 2))), jnp.zeros((tgt_x.shape + (1,))),
        ship_count, angular_velocity, max_speed, n_iter=n_iter
    )
    return aim_x, aim_y, turns


def estimate_intercept_angles(
    src_x: jnp.ndarray,
    src_y: jnp.ndarray,
    src_r: jnp.ndarray,
    tgt_x: jnp.ndarray,
    tgt_y: jnp.ndarray,
    tgt_r: jnp.ndarray,
    tgt_is_orbiting: jnp.ndarray,
    tgt_is_comet: jnp.ndarray,
    tgt_traj: jnp.ndarray,
    tgt_valid_time: jnp.ndarray,
    ship_counts: jnp.ndarray,
    angular_velocity: jnp.ndarray,
    max_speed: jnp.ndarray,
    n_iter: int = 6,
    sun_margin: float = 1.5,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, jnp.ndarray]:
    aim_x, aim_y, _turns, blocked = solve_intercept_with_wait(
        src_x, src_y, src_r, tgt_x, tgt_y, tgt_r, tgt_is_orbiting,
        tgt_is_comet, tgt_traj, tgt_valid_time,
        ship_counts, angular_velocity, max_speed, n_iter=n_iter,
        sun_margin=sun_margin,
    )
    angle = jnp.arctan2(aim_y - src_y, aim_x - src_x)
    return angle, aim_x, aim_y, blocked


def swept_pair_hit(
    ax: jnp.ndarray,
    ay: jnp.ndarray,
    bx: jnp.ndarray,
    by: jnp.ndarray,
    p0x: jnp.ndarray,
    p0y: jnp.ndarray,
    p1x: jnp.ndarray,
    p1y: jnp.ndarray,
    radius: jnp.ndarray,
) -> jnp.ndarray:
    f_min_x = jnp.minimum(ax, bx)
    f_max_x = jnp.maximum(ax, bx)
    f_min_y = jnp.minimum(ay, by)
    f_max_y = jnp.maximum(ay, by)
    
    p_min_x = jnp.minimum(p0x, p1x) - radius
    p_max_x = jnp.maximum(p0x, p1x) + radius
    p_min_y = jnp.minimum(p0y, p1y) - radius
    p_max_y = jnp.maximum(p0y, p1y) + radius
    
    intersect = (f_min_x <= p_max_x) & (f_max_x >= p_min_x) & \
                (f_min_y <= p_max_y) & (f_max_y >= p_min_y)

    d0x = ax - p0x
    d0y = ay - p0y
    dvx = (bx - ax) - (p1x - p0x)
    dvy = (by - ay) - (p1y - p0y)
    a = dvx * dvx + dvy * dvy
    b = 2.0 * (d0x * dvx + d0y * dvy)
    c = d0x * d0x + d0y * d0y - radius * radius
    disc = b * b - 4.0 * a * c
    no_motion = a < 1e-12
    hit_no_motion = c <= 0.0
    sq = jnp.sqrt(jnp.maximum(disc, 0.0))
    t1 = (-b - sq) / (2.0 * a + 1e-12)
    t2 = (-b + sq) / (2.0 * a + 1e-12)
    hit_motion = (disc >= 0.0) & (t2 >= 0.0) & (t1 <= 1.0)
    return intersect & jnp.where(no_motion, hit_no_motion, hit_motion)


def fleet_speed(ships: jnp.ndarray, max_speed: jnp.ndarray) -> jnp.ndarray:
    log_ships = jnp.log(jnp.maximum(ships, 1.0))
    log1000 = jnp.log(1000.0)
    speed = max_speed * (1.0 - 0.5 * jnp.minimum(1.0, log_ships / log1000))
    return jnp.maximum(speed, 1.0)


In [ ]:
%%writefile src/orbit_wars/reference.py
"""Reference Kaggle env bridge for reset and parity validation."""

from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path
from types import ModuleType
from typing import Any


def load_orbit_wars_module() -> ModuleType:
    """Import official orbit_wars.py (installed package or local checkout)."""
    try:
        from kaggle_environments.envs.orbit_wars import orbit_wars as ref

        return ref
    except ImportError:
        pass

    candidates = [
        Path("/media/yahor/ADATA SE880/datasets/kaggle-environments-master"),
        Path(__file__).resolve().parents[3] / "analysis" / "fast_kaggle_env",
    ]
    for root in candidates:
        module_path = root / "kaggle_environments" / "envs" / "orbit_wars" / "orbit_wars.py"
        if module_path.exists():
            if str(root) not in sys.path:
                sys.path.insert(0, str(root))
            from kaggle_environments.envs.orbit_wars import orbit_wars as ref

            return ref

    raise ImportError(
        "Could not import kaggle_environments.envs.orbit_wars.orbit_wars. "
        "On Kaggle this should be preinstalled; locally install kaggle-environments "
        "or set analysis/fast_kaggle_env."
    )


@dataclass(slots=True)
class ReferenceStep:
    observations: list[Any]
    rewards: tuple[float, float]
    done: bool


def add_env_root(env_root: str | Path) -> Path:
    path = Path(env_root)
    if not path.is_absolute():
        path = (Path.cwd() / path).resolve()
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))
    return path


def default_env_root() -> Path:
    repo = Path(__file__).resolve().parents[2]
    fast = repo / "analysis" / "fast_kaggle_env"
    official = Path("/media/yahor/ADATA SE880/datasets/kaggle-environments-master")
    if fast.exists():
        return fast
    if official.exists():
        return official
    return fast


def make_reference_env(
    *,
    seed: int,
    episode_steps: int = 500,
    env_root: str | Path | None = None,
) -> Any:
    add_env_root(env_root or default_env_root())
    from kaggle_environments import make

    configuration = {"episodeSteps": int(episode_steps), "seed": int(seed), "randomSeed": int(seed)}
    env = make("orbit_wars", configuration=configuration, debug=False)
    env.reset(num_agents=2)
    return env


def extract_observation(state: Any) -> Any:
    if isinstance(state, dict):
        return state.get("observation")
    return getattr(state, "observation")


def extract_reward(state: Any) -> float:
    if isinstance(state, dict):
        value = state.get("reward", 0.0)
    else:
        value = getattr(state, "reward", 0.0)
    return 0.0 if value is None else float(value)


def extract_status(state: Any) -> str:
    if isinstance(state, dict):
        return str(state.get("status", "UNKNOWN"))
    return str(getattr(state, "status", "UNKNOWN"))


def reference_reset(seed: int, *, episode_steps: int = 500, env_root: str | Path | None = None) -> tuple[Any, ReferenceStep]:
    env = make_reference_env(seed=seed, episode_steps=episode_steps, env_root=env_root)
    states = env.step([[], []])
    obs = [extract_observation(states[i]) for i in range(2)]
    rewards = (extract_reward(states[0]), extract_reward(states[1]))
    done = extract_status(states[0]) != "ACTIVE"
    return env, ReferenceStep(observations=obs, rewards=rewards, done=done)


def reference_step(env: Any, actions: list[list[list[float | int]]]) -> ReferenceStep:
    states = env.step(actions)
    obs = [extract_observation(states[i]) for i in range(2)]
    rewards = (extract_reward(states[0]), extract_reward(states[1]))
    done = extract_status(states[0]) != "ACTIVE"
    return ReferenceStep(observations=obs, rewards=rewards, done=done)


def episode_seed_from_env(env: Any) -> int:
    info = getattr(env, "info", None) or {}
    seed = info.get("seed")
    if seed is not None:
        return int(seed)
    return 0


In [ ]:
%%writefile src/policy.py
"""GraphTransformerV9 network.

V9 removes the all-pairs edge-aware transformer blocks from V8. The policy
trunk is a normal node transformer that remains fully trainable during PPO.
Edge facts are encoded by a tiny MLP and consumed directly by the send/target/
bucket heads. The value network is fully separate and receives a lightweight
edge summary.
"""

from __future__ import annotations

import operator

import equinox as eqx
import jax
import jax.numpy as jnp

from models.planet_transformer_jax.network_jax import (
    AttentionPool,
    EdgeEncoder,
    LaunchCountPriorNet,
    NodeEncoder,
    _ln,
    _ln_scalar,
    _wi,
)


class NodeTransformerLayer(eqx.Module):
    hidden_dim: int = eqx.field(static=True)
    heads: int = eqx.field(static=True)
    head_dim: int = eqx.field(static=True)
    scale: float = eqx.field(static=True)
    w_q: jax.Array
    b_q: jax.Array
    w_k: jax.Array
    b_k: jax.Array
    w_v: jax.Array
    b_v: jax.Array
    w_out: jax.Array
    b_out: jax.Array
    w_ffn1: jax.Array
    b_ffn1: jax.Array
    w_ffn2: jax.Array
    b_ffn2: jax.Array
    ln_w: jax.Array
    ln_b: jax.Array
    ffn_ln_w: jax.Array
    ffn_ln_b: jax.Array

    def __init__(self, hidden_dim: int, heads: int, ffn_mult: int, key):
        keys = jax.random.split(key, 8)
        H = int(hidden_dim)
        self.hidden_dim = H
        self.heads = int(heads)
        self.head_dim = H // int(heads)
        self.scale = self.head_dim ** -0.5
        self.ln_w = jnp.ones(H)
        self.ln_b = jnp.zeros(H)
        self.w_q = _wi(keys[0], (H, H))
        self.b_q = jnp.zeros(H)
        self.w_k = _wi(keys[1], (H, H))
        self.b_k = jnp.zeros(H)
        self.w_v = _wi(keys[2], (H, H))
        self.b_v = jnp.zeros(H)
        self.w_out = _wi(keys[3], (H, H))
        self.b_out = jnp.zeros(H)
        self.ffn_ln_w = jnp.ones(H)
        self.ffn_ln_b = jnp.zeros(H)
        ffn_dim = H * int(ffn_mult)
        self.w_ffn1 = _wi(keys[4], (ffn_dim, H))
        self.b_ffn1 = jnp.zeros(ffn_dim)
        self.w_ffn2 = _wi(keys[5], (H, ffn_dim))
        self.b_ffn2 = jnp.zeros(H)

    def __call__(self, node_h):
        N, H = node_h.shape
        q_norm = _ln(node_h, self.ln_w, self.ln_b)
        q = (q_norm @ self.w_q.T + self.b_q).reshape(N, self.heads, self.head_dim).transpose(1, 0, 2)
        k = (q_norm @ self.w_k.T + self.b_k).reshape(N, self.heads, self.head_dim).transpose(1, 0, 2)
        v = (q_norm @ self.w_v.T + self.b_v).reshape(N, self.heads, self.head_dim).transpose(1, 0, 2)
        scores = jnp.matmul(q, k.transpose(0, 2, 1)) * self.scale
        attn = jax.nn.softmax(scores, axis=-1)
        ctx = jnp.matmul(attn, v).transpose(1, 0, 2).reshape(N, H)
        node_h = node_h + (ctx @ self.w_out.T + self.b_out)
        h_norm = _ln(node_h, self.ffn_ln_w, self.ffn_ln_b)
        node_h = node_h + (jax.nn.silu(h_norm @ self.w_ffn1.T + self.b_ffn1) @ self.w_ffn2.T + self.b_ffn2)
        return node_h


class EdgeAwareValueNet(eqx.Module):
    value_uses_edge: bool = eqx.field(static=True, default=True)
    w_proj: jax.Array
    b_proj: jax.Array
    layers: list
    edge_encoder: EdgeEncoder
    pool: AttentionPool
    w1: jax.Array
    b1: jax.Array
    w2: jax.Array
    b2: jax.Array

    def __init__(
        self,
        hidden_dim: int,
        n_layers: int,
        heads: int,
        ffn_mult: int,
        edge_dim: int,
        edge_input_dim: int,
        key,
        node_input_dim: int = 21,
    ):
        keys = jax.random.split(key, 5 + max(n_layers, 0))
        input_dim = int(node_input_dim) + 32 + 8
        self.w_proj = _wi(keys[0], (hidden_dim, input_dim))
        self.b_proj = jnp.zeros(hidden_dim)
        self.layers = [
            NodeTransformerLayer(hidden_dim, heads, ffn_mult, key=lk)
            for lk in keys[5:]
        ]
        self.edge_encoder = EdgeEncoder(edge_dim, key=keys[1], edge_input_dim=edge_input_dim)
        self.pool = AttentionPool(hidden_dim, key=keys[2])
        self.w1 = _wi(keys[3], (hidden_dim, hidden_dim + edge_dim + 8))
        self.b1 = jnp.zeros(hidden_dim)
        self.w2 = _wi(keys[4], (1, hidden_dim))
        self.b2 = jnp.zeros(1)

    def __call__(self, node_features, future_sight, global_features, edge_features=None):
        gf = jnp.repeat(global_features[None, :], 60, axis=0)
        h = jnp.concatenate([node_features, future_sight, gf], axis=-1)
        h = jax.nn.silu(_ln(h) @ self.w_proj.T + self.b_proj)
        for layer in self.layers:
            h = layer(h)
        node_summary = self.pool(_ln(h))
        if edge_features is None:
            edge_summary = jnp.zeros(self.edge_encoder.w2.shape[0], dtype=h.dtype)
        else:
            edge_h = self.edge_encoder(edge_features)
            edge_summary = jnp.mean(edge_h, axis=(0, 1))
        z = jnp.concatenate([node_summary, edge_summary, global_features])
        z = jax.nn.silu(_ln_scalar(z) @ self.w1.T + self.b1)
        return jnp.tanh((z @ self.w2.T + self.b2).squeeze(-1))


class V9PolicyHead(eqx.Module):
    w_send1: jax.Array
    b_send1: jax.Array
    w_send2: jax.Array
    b_send2: jax.Array
    w_tgt1: jax.Array
    b_tgt1: jax.Array
    w_tgt2: jax.Array
    b_tgt2: jax.Array
    w_bucket1: jax.Array
    b_bucket1: jax.Array
    w_bucket2: jax.Array
    b_bucket2: jax.Array
    w_frac1: jax.Array
    b_frac1: jax.Array
    w_frac2: jax.Array
    b_frac2: jax.Array
    n_ship_options: int = eqx.field(static=True)

    def __init__(self, hidden_dim: int, edge_dim: int, n_ship_options: int = 3, key=None):
        k0, k1, k2, k3, k4, k5, k6, k7 = jax.random.split(key, 8)
        H, E = int(hidden_dim), int(edge_dim)
        self.n_ship_options = int(n_ship_options)
        send_d = H + 8
        pair_d = 2 * H + E + 8
        bucket_raw_d = 4
        bucket_d = pair_d + bucket_raw_d
        self.w_send1 = _wi(k0, (H, send_d))
        self.b_send1 = jnp.zeros(H)
        self.w_send2 = _wi(k1, (1, H))
        self.b_send2 = jnp.zeros(1)
        self.w_tgt1 = _wi(k2, (H, pair_d))
        self.b_tgt1 = jnp.zeros(H)
        self.w_tgt2 = _wi(k3, (1, H))
        self.b_tgt2 = jnp.zeros(1)
        self.w_bucket1 = _wi(k4, (H, bucket_d))
        self.b_bucket1 = jnp.zeros(H)
        self.w_bucket2 = _wi(k5, (1, H))
        self.b_bucket2 = jnp.zeros(1)
        self.w_frac1 = _wi(k6, (H, bucket_d))
        self.b_frac1 = jnp.zeros(H)
        self.w_frac2 = _wi(k7, (1, H))
        self.b_frac2 = jnp.zeros(1)

    def _bucket_raw(self, raw_edge_s):
        roi = raw_edge_s[:, :, 6:7]
        ratio = raw_edge_s[:, :, 7:8]
        turns = raw_edge_s[:, :, 8:11]
        clear = raw_edge_s[:, :, 11:14]
        per_bucket = [
            jnp.concatenate([roi, ratio, turns[:, :, i:i + 1], clear[:, :, i:i + 1]], axis=-1)
            for i in range(self.n_ship_options)
        ]
        return jnp.stack(per_bucket, axis=2).reshape(
            raw_edge_s.shape[0], raw_edge_s.shape[1], self.n_ship_options, 4
        )

    def __call__(self, node_h, edge_h, raw_edge, owned, gf):
        N, H = node_h.shape
        E = edge_h.shape[-1]
        src_safe = jnp.where(owned >= 0, owned, 0)
        src_h = node_h[src_safe]
        edge_s = edge_h[src_safe]
        raw_edge_s = raw_edge[src_safe]
        M = owned.shape[0]

        send_cat = jnp.concatenate([src_h, jnp.broadcast_to(gf[None, :], (M, 8))], axis=-1)
        send = (jax.nn.silu(send_cat @ self.w_send1.T + self.b_send1) @ self.w_send2.T + self.b_send2).squeeze(-1)

        pair_cat = jnp.concatenate(
            [
                jnp.broadcast_to(src_h[:, None, :], (M, N, H)),
                jnp.broadcast_to(node_h[None, :, :], (M, N, H)),
                edge_s,
                jnp.broadcast_to(gf[None, None, :], (M, N, 8)),
            ],
            axis=-1,
        )
        flat_pair = pair_cat.reshape(-1, 2 * H + E + 8)
        target_base = (
            jax.nn.silu(flat_pair @ self.w_tgt1.T + self.b_tgt1) @ self.w_tgt2.T + self.b_tgt2
        ).reshape(M, N)

        bucket_raw = self._bucket_raw(raw_edge_s)
        bucket_cat = jnp.concatenate(
            [
                jnp.broadcast_to(pair_cat[:, :, None, :], (M, N, self.n_ship_options, 2 * H + E + 8)),
                bucket_raw,
            ],
            axis=-1,
        )
        flat_bucket = bucket_cat.reshape(-1, 2 * H + E + 8 + 4)
        bucket_utility = (
            jax.nn.silu(flat_bucket @ self.w_bucket1.T + self.b_bucket1) @ self.w_bucket2.T + self.b_bucket2
        ).reshape(M, N, self.n_ship_options)
        target = target_base + jnp.max(bucket_utility, axis=-1)
        frac = (
            jax.nn.silu(flat_bucket @ self.w_frac1.T + self.b_frac1) @ self.w_frac2.T + self.b_frac2
        ).reshape(M, N, self.n_ship_options)
        return send, target, frac


class GraphTransformerV9(eqx.Module):
    value_uses_edge: bool = eqx.field(static=True, default=True)
    node_encoder: NodeEncoder
    edge_encoder: EdgeEncoder
    layers: list
    policy_heads: tuple
    value_head: EdgeAwareValueNet
    launch_prior_head: LaunchCountPriorNet
    node_ln_w: jax.Array
    node_ln_b: jax.Array
    edge_ln_w: jax.Array
    edge_ln_b: jax.Array
    n_policy_heads: int = eqx.field(static=True)
    edge_dim: int = eqx.field(static=True)

    def __init__(
        self,
        hidden_dim: int = 64,
        n_layers: int = 5,
        heads: int = 4,
        ffn_mult: int = 2,
        value_hidden_dim: int = 32,
        value_layers: int = 4,
        prior_hidden_dim: int = 32,
        n_ship_options: int = 3,
        node_input_dim: int = 21,
        edge_input_dim: int = 14,
        edge_dim: int = 16,
        n_policy_heads: int = 10,
        key=None,
    ):
        if key is None:
            key = jax.random.PRNGKey(0)
        k1, k2, k3, k4, k5, k6 = jax.random.split(key, 6)
        self.edge_dim = int(edge_dim)
        self.node_encoder = NodeEncoder(hidden_dim, key=k1, node_input_dim=node_input_dim)
        self.edge_encoder = EdgeEncoder(edge_dim, key=k2, edge_input_dim=edge_input_dim)
        self.layers = [
            NodeTransformerLayer(hidden_dim, heads, ffn_mult, key=lk)
            for lk in jax.random.split(k3, n_layers)
        ]
        self.node_ln_w = jnp.ones(hidden_dim)
        self.node_ln_b = jnp.zeros(hidden_dim)
        self.edge_ln_w = jnp.ones(edge_dim)
        self.edge_ln_b = jnp.zeros(edge_dim)
        self.n_policy_heads = int(n_policy_heads)
        self.policy_heads = tuple(
            V9PolicyHead(hidden_dim, edge_dim, n_ship_options=n_ship_options, key=hk)
            for hk in jax.random.split(k4, self.n_policy_heads)
        )
        self.value_head = EdgeAwareValueNet(
            value_hidden_dim,
            value_layers,
            max(1, min(heads, value_hidden_dim)),
            ffn_mult,
            edge_dim,
            edge_input_dim,
            key=k5,
            node_input_dim=node_input_dim,
        )
        self.launch_prior_head = LaunchCountPriorNet(prior_hidden_dim, key=k6)

    def encode(self, node_features, edge_features, future_sight, global_features):
        node_h = self.node_encoder(node_features, future_sight, global_features)
        for layer in self.layers:
            node_h = layer(node_h)
        node_h = _ln(node_h, self.node_ln_w, self.node_ln_b)
        edge_h = self.edge_encoder(edge_features)
        eflat = edge_h.reshape(-1, edge_h.shape[-1])
        edge_h = _ln(eflat, self.edge_ln_w, self.edge_ln_b).reshape(edge_h.shape)
        return node_h, edge_h

    def __call__(
        self,
        node_features,
        edge_features,
        future_sight,
        global_features,
        owned_nodes,
        player_head_idx=0,
    ):
        node_h, edge_h = self.encode(node_features, edge_features, future_sight, global_features)
        if isinstance(player_head_idx, int):
            head_idx = max(0, min(operator.index(player_head_idx), self.n_policy_heads - 1))
            send, tgt, frac = self.policy_heads[head_idx](node_h, edge_h, edge_features, owned_nodes, global_features)
        else:
            idx = jnp.clip(jnp.asarray(player_head_idx, dtype=jnp.int32), 0, self.n_policy_heads - 1)

            def branch(head):
                return lambda _: head(node_h, edge_h, edge_features, owned_nodes, global_features)

            send, tgt, frac = jax.lax.switch(
                idx,
                tuple(branch(head) for head in self.policy_heads),
                operand=None,
            )
        val = self.value_head(node_features, future_sight, global_features, edge_features)
        prior_mu_log, prior_sigma_log = self.launch_prior_head(
            node_features, future_sight, global_features
        )
        return send, tgt, frac, prior_mu_log, prior_sigma_log, val


In [ ]:
%%writefile src/ppo.py
import jax
import jax.numpy as jnp
import optax
import equinox as eqx
from typing import NamedTuple, Any

from .policy import GraphTransformerV9
from .env import OrbitWarsPureJaxEnv, compute_reward
from .orbit_wars.features_jax import ObsBatch, extract_obs_v9_jax

MAX_PPO_LAUNCH_SLOTS = 16
N_SHIP_OPTIONS = 3


class Transition(NamedTuple):
    done: jnp.ndarray
    value: jnp.ndarray
    reward: jnp.ndarray
    obs: ObsBatch
    action_tgt: jnp.ndarray
    action_frac: jnp.ndarray
    log_prob: jnp.ndarray


class TrainState(NamedTuple):
    model: GraphTransformerV9
    opt_state: optax.OptState


def _normalized_entropy(probs, log_probs, valid_mask):
    """Return entropy normalized by the valid categorical support size."""
    raw_ent = -jnp.sum(probs * log_probs, axis=-1)
    valid_count = jnp.sum(valid_mask.astype(jnp.float32), axis=-1)
    denom = jnp.where(valid_count > 1.0, jnp.log(valid_count), 1.0)
    return jnp.where(valid_count > 1.0, raw_ent / denom, 0.0)


def _weighted_launch_entropy(tgt_probs, tgt_log_probs, tgt_mask, frac_probs, frac_log_probs, frac_mask):
    target_weight = 1.0
    frac_weight = 0.35
    tgt_ent = _normalized_entropy(tgt_probs, tgt_log_probs, tgt_mask)
    frac_ent = _normalized_entropy(frac_probs, frac_log_probs, frac_mask)
    return target_weight * tgt_ent + frac_weight * frac_ent


def _send_logit_bias():
    return 0.0


def forward_and_sample_v9_compact(
    model: GraphTransformerV9,
    obs: ObsBatch,
    rng: jnp.ndarray,
    temperature: jnp.ndarray = 1.0,
) -> tuple:
    """V9 rollout sampler that avoids no-launch target/fraction heads."""
    node_h, edge_h = model.encode(
        obs.node_features,
        obs.edge_features,
        obs.future_sight,
        obs.global_features,
    )
    head = model.policy_heads[0]

    M = obs.owned_nodes.shape[0]
    N = obs.node_features.shape[0]
    B = head.n_ship_options
    H = node_h.shape[-1]
    E = edge_h.shape[-1]

    src_safe = jnp.where(obs.owned_nodes >= 0, obs.owned_nodes, 0)
    slot_valid = obs.owned_nodes >= 0
    src_h = node_h[src_safe]
    send_cat = jnp.concatenate(
        [src_h, jnp.broadcast_to(obs.global_features[None, :], (M, 8))],
        axis=-1,
    )
    send_logits = (
        jax.nn.silu(send_cat @ head.w_send1.T + head.b_send1) @ head.w_send2.T + head.b_send2
    ).squeeze(-1) + _send_logit_bias()
    send_logits_scaled = send_logits / jnp.maximum(temperature, 1e-4)

    tgt_valid = obs.edge_valid_mask.any(axis=-1)
    target_ids = jnp.arange(N)[None, :]
    non_noop_target = target_ids != src_safe[:, None]
    combined_tgt_mask = tgt_valid & slot_valid[:, None] & non_noop_target
    slot_launch_feasible = slot_valid & combined_tgt_mask.any(axis=-1)

    rng_send, rng_tgt, rng_frac = jax.random.split(rng, 3)
    send_pair_logits = jnp.stack([jnp.zeros_like(send_logits_scaled), send_logits_scaled], axis=-1)
    send_pair_logits = jnp.where(
        slot_launch_feasible[:, None] | (jnp.arange(2)[None, :] == 0),
        send_pair_logits,
        -1e9,
    )
    raw_send = jax.vmap(lambda logits, r: jax.random.categorical(r, logits))(
        send_pair_logits, jax.random.split(rng_send, M)
    ).astype(jnp.int32)

    raw_launch = (raw_send == 1) & slot_launch_feasible
    launch_rank = jnp.where(raw_launch, jnp.arange(M), M + jnp.arange(M))
    launch_slots = jnp.sort(launch_rank)[:MAX_PPO_LAUNCH_SLOTS]
    launch_present = launch_slots < M
    safe_slots = jnp.where(launch_present, launch_slots, 0)
    selected_launch = (
        jnp.zeros((M,), dtype=jnp.int32)
        .at[safe_slots]
        .add(launch_present.astype(jnp.int32))
    ) > 0
    action_send = selected_launch.astype(jnp.int32)

    src_nodes = src_safe[safe_slots]
    src_h_k = node_h[src_nodes]
    edge_s = edge_h[src_nodes]
    raw_edge_s = obs.edge_features[src_nodes]
    pair_cat = jnp.concatenate(
        [
            jnp.broadcast_to(src_h_k[:, None, :], (MAX_PPO_LAUNCH_SLOTS, N, H)),
            jnp.broadcast_to(node_h[None, :, :], (MAX_PPO_LAUNCH_SLOTS, N, H)),
            edge_s,
            jnp.broadcast_to(obs.global_features[None, None, :], (MAX_PPO_LAUNCH_SLOTS, N, 8)),
        ],
        axis=-1,
    )
    flat_pair = pair_cat.reshape(-1, 2 * H + E + 8)
    target_base = (
        jax.nn.silu(flat_pair @ head.w_tgt1.T + head.b_tgt1) @ head.w_tgt2.T + head.b_tgt2
    ).reshape(MAX_PPO_LAUNCH_SLOTS, N)

    bucket_raw = head._bucket_raw(raw_edge_s)
    bucket_cat = jnp.concatenate(
        [
            jnp.broadcast_to(pair_cat[:, :, None, :], (MAX_PPO_LAUNCH_SLOTS, N, B, 2 * H + E + 8)),
            bucket_raw,
        ],
        axis=-1,
    )
    flat_bucket = bucket_cat.reshape(-1, 2 * H + E + 8 + 4)
    bucket_utility = (
        jax.nn.silu(flat_bucket @ head.w_bucket1.T + head.b_bucket1) @ head.w_bucket2.T + head.b_bucket2
    ).reshape(MAX_PPO_LAUNCH_SLOTS, N, B)
    target_logits = (target_base + jnp.max(bucket_utility, axis=-1)) / jnp.maximum(temperature, 1e-4)
    masked_tgt_logits = jnp.where(combined_tgt_mask[safe_slots] & launch_present[:, None], target_logits, -1e9)
    safe_tgt_logits = jnp.where(
        launch_present[:, None],
        masked_tgt_logits,
        jnp.where(jnp.arange(N)[None, :] == 0, 0.0, -1e9),
    )
    action_tgt_k = jax.vmap(lambda logits, r: jax.random.categorical(r, logits))(
        safe_tgt_logits, jax.random.split(rng_tgt, MAX_PPO_LAUNCH_SLOTS)
    )

    pair_chosen = pair_cat[jnp.arange(MAX_PPO_LAUNCH_SLOTS), action_tgt_k]
    bucket_raw_chosen = bucket_raw[jnp.arange(MAX_PPO_LAUNCH_SLOTS), action_tgt_k]
    frac_cat = jnp.concatenate(
        [
            jnp.broadcast_to(pair_chosen[:, None, :], (MAX_PPO_LAUNCH_SLOTS, B, 2 * H + E + 8)),
            bucket_raw_chosen,
        ],
        axis=-1,
    )
    flat_frac = frac_cat.reshape(-1, 2 * H + E + 8 + 4)
    frac_logits = (
        jax.nn.silu(flat_frac @ head.w_frac1.T + head.b_frac1) @ head.w_frac2.T + head.b_frac2
    ).reshape(MAX_PPO_LAUNCH_SLOTS, B)
    frac_logits = frac_logits / jnp.maximum(temperature, 1e-4)
    frac_valid = obs.edge_valid_mask[safe_slots, action_tgt_k]
    masked_frac_logits = jnp.where(frac_valid & launch_present[:, None], frac_logits, -1e9)
    safe_frac_logits = jnp.where(
        launch_present[:, None],
        masked_frac_logits,
        jnp.where(jnp.arange(B)[None, :] == 0, 0.0, -1e9),
    )
    action_frac_k = jax.vmap(lambda logits, r: jax.random.categorical(r, logits))(
        safe_frac_logits, jax.random.split(rng_frac, MAX_PPO_LAUNCH_SLOTS)
    )

    encoded_tgt = (
        jnp.zeros((M,), dtype=jnp.int32)
        .at[safe_slots]
        .add(jnp.where(launch_present, action_tgt_k + 1, 0))
    )
    encoded_frac = (
        jnp.zeros((M,), dtype=jnp.int32)
        .at[safe_slots]
        .add(jnp.where(launch_present, action_frac_k + 1, 0))
    )
    action_tgt = jnp.where(encoded_tgt > 0, encoded_tgt - 1, -1)
    action_frac = jnp.where(encoded_frac > 0, encoded_frac - 1, 0)

    send_log_probs = jax.nn.log_softmax(send_pair_logits, axis=-1)
    chosen_send_lp = send_log_probs[jnp.arange(M), action_send]
    tgt_log_probs = jax.nn.log_softmax(masked_tgt_logits, axis=-1)
    frac_log_probs = jax.nn.log_softmax(masked_frac_logits, axis=-1)
    chosen_tgt_lp_k = tgt_log_probs[jnp.arange(MAX_PPO_LAUNCH_SLOTS), action_tgt_k]
    chosen_frac_lp_k = frac_log_probs[jnp.arange(MAX_PPO_LAUNCH_SLOTS), action_frac_k]
    launched_extra_lp = jnp.zeros((M,), dtype=jnp.float32).at[safe_slots].add(
        jnp.where(launch_present, chosen_tgt_lp_k + chosen_frac_lp_k, 0.0)
    )

    num_owned = jnp.maximum(jnp.sum(slot_valid), 1.0)
    step_lp = jnp.sum(jnp.where(slot_valid, chosen_send_lp + launched_extra_lp, 0.0)) / num_owned

    tgt_probs = jax.nn.softmax(masked_tgt_logits, axis=-1)
    frac_probs = jax.nn.softmax(masked_frac_logits, axis=-1)
    launch_ent_k = _weighted_launch_entropy(
        tgt_probs,
        tgt_log_probs,
        combined_tgt_mask[safe_slots] & launch_present[:, None],
        frac_probs,
        frac_log_probs,
        frac_valid & launch_present[:, None],
    )
    launch_ent = jnp.zeros((M,), dtype=jnp.float32).at[safe_slots].add(
        jnp.where(launch_present, launch_ent_k, 0.0)
    )
    entropy = jnp.sum(jnp.where(slot_valid, launch_ent, 0.0)) / num_owned

    value = model.value_head(obs.node_features, obs.future_sight, obs.global_features, obs.edge_features)
    action_tgt = jnp.where(slot_valid & (action_send == 1), action_tgt, -1)
    action_frac = jnp.where(slot_valid & (action_send == 1), action_frac, 0)
    return action_tgt, action_frac, step_lp, value.squeeze(), entropy


def compute_log_prob_v9_compact(
    model: GraphTransformerV9,
    obs: ObsBatch,
    action_tgt: jnp.ndarray,
    action_frac: jnp.ndarray,
    temperature: jnp.ndarray = 1.0,
    apm_expected_temperature: jnp.ndarray = 1.0,
) -> tuple:
    """Exact PPO log-prob for taken actions without all-slot pair heads."""
    node_h, edge_h = model.encode(
        obs.node_features,
        obs.edge_features,
        obs.future_sight,
        obs.global_features,
    )
    head = model.policy_heads[0]

    M = obs.owned_nodes.shape[0]
    N = obs.node_features.shape[0]
    B = head.n_ship_options
    H = node_h.shape[-1]
    E = edge_h.shape[-1]

    src_safe = jnp.where(obs.owned_nodes >= 0, obs.owned_nodes, 0)
    slot_valid = obs.owned_nodes >= 0
    src_h = node_h[src_safe]
    send_cat = jnp.concatenate(
        [src_h, jnp.broadcast_to(obs.global_features[None, :], (M, 8))],
        axis=-1,
    )
    send_logits = (
        jax.nn.silu(send_cat @ head.w_send1.T + head.b_send1) @ head.w_send2.T + head.b_send2
    ).squeeze(-1) + _send_logit_bias()
    send_logits_scaled = send_logits / jnp.maximum(temperature, 1e-4)

    tgt_valid = obs.edge_valid_mask.any(axis=-1)
    target_ids = jnp.arange(N)[None, :]
    non_noop_target = target_ids != src_safe[:, None]
    combined_tgt_mask = tgt_valid & slot_valid[:, None] & non_noop_target
    slot_launch_feasible = slot_valid & combined_tgt_mask.any(axis=-1)

    raw_action_send = action_tgt >= 0
    action_send = raw_action_send & slot_launch_feasible
    send_pair_logits = jnp.stack([jnp.zeros_like(send_logits_scaled), send_logits_scaled], axis=-1)
    send_pair_logits = jnp.where(
        slot_launch_feasible[:, None] | (jnp.arange(2)[None, :] == 0),
        send_pair_logits,
        -1e9,
    )
    send_log_probs = jax.nn.log_softmax(send_pair_logits, axis=-1)
    send_probs = jax.nn.softmax(send_pair_logits, axis=-1)
    chosen_send_lp = send_log_probs[jnp.arange(M), action_send.astype(jnp.int32)]

    launch_rank = jnp.where(action_send, jnp.arange(M), M + jnp.arange(M))
    launch_slots = jnp.sort(launch_rank)[:MAX_PPO_LAUNCH_SLOTS]
    launch_present = launch_slots < M
    safe_slots = jnp.where(launch_present, launch_slots, 0)
    safe_tgt = jnp.clip(action_tgt[safe_slots], 0, N - 1)
    safe_frac = jnp.clip(action_frac[safe_slots], 0, B - 1)

    src_nodes = src_safe[safe_slots]
    src_h_k = node_h[src_nodes]
    edge_s = edge_h[src_nodes]
    raw_edge_s = obs.edge_features[src_nodes]

    pair_cat = jnp.concatenate(
        [
            jnp.broadcast_to(src_h_k[:, None, :], (MAX_PPO_LAUNCH_SLOTS, N, H)),
            jnp.broadcast_to(node_h[None, :, :], (MAX_PPO_LAUNCH_SLOTS, N, H)),
            edge_s,
            jnp.broadcast_to(obs.global_features[None, None, :], (MAX_PPO_LAUNCH_SLOTS, N, 8)),
        ],
        axis=-1,
    )
    flat_pair = pair_cat.reshape(-1, 2 * H + E + 8)
    target_base = (
        jax.nn.silu(flat_pair @ head.w_tgt1.T + head.b_tgt1) @ head.w_tgt2.T + head.b_tgt2
    ).reshape(MAX_PPO_LAUNCH_SLOTS, N)

    bucket_raw = head._bucket_raw(raw_edge_s)
    bucket_cat = jnp.concatenate(
        [
            jnp.broadcast_to(pair_cat[:, :, None, :], (MAX_PPO_LAUNCH_SLOTS, N, B, 2 * H + E + 8)),
            bucket_raw,
        ],
        axis=-1,
    )
    flat_bucket = bucket_cat.reshape(-1, 2 * H + E + 8 + 4)
    bucket_utility = (
        jax.nn.silu(flat_bucket @ head.w_bucket1.T + head.b_bucket1) @ head.w_bucket2.T + head.b_bucket2
    ).reshape(MAX_PPO_LAUNCH_SLOTS, N, B)
    target_logits = (target_base + jnp.max(bucket_utility, axis=-1)) / jnp.maximum(temperature, 1e-4)

    target_mask = combined_tgt_mask[safe_slots]
    masked_tgt_logits = jnp.where(target_mask & launch_present[:, None], target_logits, -1e9)
    tgt_log_probs = jax.nn.log_softmax(masked_tgt_logits, axis=-1)
    chosen_tgt_lp_k = tgt_log_probs[jnp.arange(MAX_PPO_LAUNCH_SLOTS), safe_tgt]

    pair_chosen = pair_cat[jnp.arange(MAX_PPO_LAUNCH_SLOTS), safe_tgt]
    bucket_raw_chosen = bucket_raw[jnp.arange(MAX_PPO_LAUNCH_SLOTS), safe_tgt]
    frac_cat = jnp.concatenate(
        [
            jnp.broadcast_to(pair_chosen[:, None, :], (MAX_PPO_LAUNCH_SLOTS, B, 2 * H + E + 8)),
            bucket_raw_chosen,
        ],
        axis=-1,
    )
    flat_frac = frac_cat.reshape(-1, 2 * H + E + 8 + 4)
    frac_logits = (
        jax.nn.silu(flat_frac @ head.w_frac1.T + head.b_frac1) @ head.w_frac2.T + head.b_frac2
    ).reshape(MAX_PPO_LAUNCH_SLOTS, B)
    frac_logits = frac_logits / jnp.maximum(temperature, 1e-4)
    frac_valid = obs.edge_valid_mask[safe_slots, safe_tgt]
    masked_frac_logits = jnp.where(frac_valid & launch_present[:, None], frac_logits, -1e9)
    frac_log_probs = jax.nn.log_softmax(masked_frac_logits, axis=-1)
    chosen_frac_lp_k = frac_log_probs[jnp.arange(MAX_PPO_LAUNCH_SLOTS), safe_frac]

    launched_extra_lp = jnp.zeros((M,), dtype=jnp.float32).at[safe_slots].add(
        jnp.where(launch_present, chosen_tgt_lp_k + chosen_frac_lp_k, 0.0)
    )
    num_owned = jnp.maximum(jnp.sum(slot_valid), 1.0)
    step_lp = jnp.sum(jnp.where(slot_valid, chosen_send_lp + launched_extra_lp, 0.0)) / num_owned

    p_launch_policy = jnp.where(slot_launch_feasible, send_probs[:, 1], 0.0)

    frac_probs = jax.nn.softmax(masked_frac_logits, axis=-1)
    tgt_probs = jax.nn.softmax(masked_tgt_logits, axis=-1)
    launch_ent_k = _weighted_launch_entropy(
        tgt_probs,
        tgt_log_probs,
        target_mask & launch_present[:, None],
        frac_probs,
        frac_log_probs,
        frac_valid & launch_present[:, None],
    )
    launch_ent = jnp.zeros((M,), dtype=jnp.float32).at[safe_slots].add(
        jnp.where(launch_present, launch_ent_k, 0.0)
    )
    entropy = jnp.sum(jnp.where(slot_valid, launch_ent, 0.0)) / num_owned

    value = model.value_head(obs.node_features, obs.future_sight, obs.global_features, obs.edge_features)
    prior_mu_log, prior_sigma_log = model.launch_prior_head(
        obs.node_features, obs.future_sight, obs.global_features
    )
    return (
        step_lp,
        value.squeeze(),
        entropy,
        p_launch_policy,
        prior_mu_log,
        prior_sigma_log,
    )


def actions_to_env_format(action_tgt: jnp.ndarray, action_frac: jnp.ndarray) -> jnp.ndarray:
    """Convert target and ship-bin actions to flat action indices for jax_orbit_wars_step."""
    flat = action_tgt * N_SHIP_OPTIONS + action_frac
    flat = jnp.where(action_tgt >= 0, flat, -1)
    return flat.astype(jnp.int32)


def make_train(config):
    config["NUM_UPDATES"] = config["TOTAL_TIMESTEPS"] // config["NUM_STEPS"] // config["NUM_ENVS"]
    config["MINIBATCH_SIZE"] = config["NUM_ENVS"] * config["NUM_STEPS"] // config["NUM_MINIBATCHES"]
    
    env = OrbitWarsPureJaxEnv(
        episode_steps=config.get("EPISODE_STEPS", 500),
        ship_speed=config.get("SHIP_SPEED", 6.0)
    )

    def linear_schedule(count):
        frac = 1.0 - (count // (config["NUM_MINIBATCHES"] * config["UPDATE_EPOCHS"])) / config["NUM_UPDATES"]
        return config["LR"] * jnp.maximum(frac, 0.0)

    if config.get("ANNEAL_LR", True):
        tx = optax.chain(
            optax.clip_by_global_norm(config["MAX_GRAD_NORM"]),
            optax.adam(learning_rate=linear_schedule, eps=1e-5),
        )
    else:
        tx = optax.chain(
            optax.clip_by_global_norm(config["MAX_GRAD_NORM"]),
            optax.adam(config["LR"], eps=1e-5),
        )

    def init_fn(rng):
        rng, _rng = jax.random.split(rng)
        
        # Initialize Equinox model
        model = GraphTransformerV9(
            hidden_dim=config.get("D_MODEL", 64),
            n_layers=config.get("NUM_LAYERS", 5),
            heads=config.get("NUM_HEADS", 4),
            n_ship_options=3,
            node_input_dim=21,
            edge_input_dim=14,
            edge_dim=16,
            n_policy_heads=1,
            key=_rng
        )
        
        opt_state = tx.init(eqx.filter(model, eqx.is_array))
        
        train_state = TrainState(
            model=model,
            opt_state=opt_state
        )

        # INIT ENV
        rng, _rng = jax.random.split(rng)
        reset_rng = jax.random.split(_rng, config["NUM_ENVS"])
        obsv, env_state = jax.vmap(env.reset)(reset_rng)

        return (train_state, env_state, obsv, rng)

    def update_fn(runner_state, unused=None):
            
            # COLLECT TRAJECTORIES
            def _env_step(runner_state, unused):
                train_state, env_state, last_obs, rng = runner_state

                # SELECT ACTION FOR PLAYER 0
                sample_fn = jax.vmap(forward_and_sample_v9_compact, in_axes=(None, 0, 0, None))
                rng, rng_sample = jax.random.split(rng)
                sample_keys = jax.random.split(rng_sample, config["NUM_ENVS"])
                action_tgt, action_frac, step_lp, value, entropy = sample_fn(
                    train_state.model, last_obs, sample_keys, 1.0
                )
                
                # Format to JAX env actions
                actions_p0 = actions_to_env_format(action_tgt, action_frac)
                owned_p0 = last_obs.owned_nodes
                
                # Opponent is no-op
                actions_p1 = jnp.full(actions_p0.shape, -1, dtype=jnp.int32)
                owned_p1 = jnp.full(owned_p0.shape, -1, dtype=jnp.int32)

                # STEP ENV
                rng, _rng = jax.random.split(rng)
                step_rngs = jax.random.split(_rng, config["NUM_ENVS"])
                
                obsv, env_state, reward, done, info = jax.vmap(env.step)(
                    step_rngs, env_state, actions_p0, owned_p0, actions_p1, owned_p1
                )
                
                transition = Transition(
                    done=done,
                    value=value,
                    reward=reward,
                    obs=last_obs,
                    action_tgt=action_tgt,
                    action_frac=action_frac,
                    log_prob=step_lp,
                )
                
                runner_state = (train_state, env_state, obsv, rng)
                return runner_state, transition

            runner_state, traj_batch = jax.lax.scan(_env_step, runner_state, None, config["NUM_STEPS"])

            # CALCULATE ADVANTAGE
            train_state, env_state, last_obs, rng = runner_state
            
            # Get last value estimate
            sample_fn = jax.vmap(forward_and_sample_v9_compact, in_axes=(None, 0, 0, None))
            rng, rng_sample = jax.random.split(rng)
            sample_keys = jax.random.split(rng_sample, config["NUM_ENVS"])
            _, _, _, last_val, _ = sample_fn(
                train_state.model, last_obs, sample_keys, 1.0
            )

            def _calculate_gae(traj_batch, last_val):
                def _get_advantages(gae_and_next_value, transition):
                    gae, next_value = gae_and_next_value
                    done, value, reward = transition.done, transition.value, transition.reward
                    delta = reward + config["GAMMA"] * next_value * (1 - done) - value
                    gae = delta + config["GAMMA"] * config["GAE_LAMBDA"] * (1 - done) * gae
                    return (gae, value), gae

                _, advantages = jax.lax.scan(
                    _get_advantages,
                    (jnp.zeros_like(last_val), last_val),
                    traj_batch,
                    reverse=True,
                    unroll=16,
                )
                return advantages, advantages + traj_batch.value

            advantages, targets = _calculate_gae(traj_batch, last_val)

            # UPDATE NETWORK
            def _update_epoch(update_state, unused):
                def _update_minbatch(train_state, batch_info):
                    traj_batch, advantages, targets = batch_info

                    def _loss_fn(model, traj_batch, gae, targets):
                        def loss_single(obs, act_tgt, act_frac, old_lp, adv, target):
                            step_lp, val, ent, _prior_lp, _mu, _sig = compute_log_prob_v9_compact(
                                model, obs, act_tgt, act_frac, 1.0, 1.0
                            )
                            
                            log_ratio = jnp.clip(step_lp - old_lp, -20.0, 20.0)
                            ratio = jnp.exp(log_ratio)
                            surr1 = ratio * adv
                            surr2 = jnp.clip(ratio, 1.0 - config["CLIP_EPS"], 1.0 + config["CLIP_EPS"]) * adv
                            loss_actor = -jnp.minimum(surr1, surr2)
                            
                            value_loss = 0.5 * jnp.square(val - target)
                            
                            approx_kl = ratio - 1.0 - log_ratio
                            clip_frac = (jnp.abs(ratio - 1.0) > config["CLIP_EPS"]).astype(jnp.float32)
                            
                            return loss_actor + config["VF_COEF"] * value_loss - config["ENT_COEF"] * ent, (value_loss, loss_actor, ent, approx_kl, clip_frac)
                            
                        losses, aux = jax.vmap(loss_single)(
                            traj_batch.obs,
                            traj_batch.action_tgt,
                            traj_batch.action_frac,
                            traj_batch.log_prob,
                            gae,
                            targets
                        )
                        
                        metrics = (aux[0].mean(), aux[1].mean(), aux[2].mean(), aux[3].mean(), aux[4].mean())
                        return losses.mean(), metrics

                    def _loss_fn_wrapper(model_params, model_static, traj_batch, gae, targets):
                        model = eqx.combine(model_params, model_static)
                        return _loss_fn(model, traj_batch, gae, targets)

                    # GAE Normalization
                    gae = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

                    model_params, model_static = eqx.partition(train_state.model, eqx.is_array)
                    
                    grad_fn = jax.value_and_grad(_loss_fn_wrapper, has_aux=True)
                    (total_loss, metrics), grads = grad_fn(model_params, model_static, traj_batch, gae, targets)
                    
                    updates, opt_state = tx.update(grads, train_state.opt_state, model_params)
                    model = eqx.apply_updates(train_state.model, updates)
                    
                    new_train_state = TrainState(model=model, opt_state=opt_state)
                    return new_train_state, (total_loss, metrics)

                train_state, traj_batch, advantages, targets, rng = update_state
                rng, _rng = jax.random.split(rng)
                
                batch_size = config["MINIBATCH_SIZE"] * config["NUM_MINIBATCHES"]
                permutation = jax.random.permutation(_rng, batch_size)
                batch = (traj_batch, advantages, targets)
                batch = jax.tree_util.tree_map(lambda x: x.reshape((batch_size,) + x.shape[2:]), batch)
                shuffled_batch = jax.tree_util.tree_map(lambda x: jnp.take(x, permutation, axis=0), batch)
                minibatches = jax.tree_util.tree_map(
                    lambda x: jnp.reshape(x, [config["NUM_MINIBATCHES"], -1] + list(x.shape[1:])),
                    shuffled_batch,
                )
                
                train_state, minibatches_metrics = jax.lax.scan(_update_minbatch, train_state, minibatches)
                
                # Average metrics over minibatches
                mean_metrics = jax.tree_util.tree_map(lambda x: x.mean(), minibatches_metrics)
                
                update_state = (train_state, traj_batch, advantages, targets, rng)
                return update_state, mean_metrics

            update_state = (train_state, traj_batch, advantages, targets, rng)
            update_state, loss_info = jax.lax.scan(_update_epoch, update_state, None, config["UPDATE_EPOCHS"])
            train_state = update_state[0]
            rng = update_state[-1]

            # Average metrics over epochs
            mean_loss_info = jax.tree_util.tree_map(lambda x: x.mean(), loss_info)
            
            # Combine metrics
            step_metrics = {
                "loss": mean_loss_info[0],
                "value_loss": mean_loss_info[1][0],
                "policy_loss": mean_loss_info[1][1],
                "entropy": mean_loss_info[1][2],
                "approx_kl": mean_loss_info[1][3],
                "clip_frac": mean_loss_info[1][4],
                "explained_variance": jnp.array(0.0), # Statically 0.0 or compute it if needed
                "reward": traj_batch.reward.mean(),
            }

            runner_state = (train_state, env_state, last_obs, rng)
            return runner_state, step_metrics

    return init_fn, update_fn


In [ ]:
%%writefile src/env.py
from __future__ import annotations

import jax
import jax.numpy as jnp
from typing import Any

from env.jax_orbit_wars import JaxEnvState, jax_orbit_wars_reset, jax_orbit_wars_step
from .orbit_wars.features_jax import ObsBatch, extract_obs_v9_jax


def compute_reward(
    state_before: JaxEnvState,
    state_after: JaxEnvState,
    player_id: jnp.ndarray,
    cfg: dict,
) -> jnp.ndarray:
    """Computes shaped reward for a player after one step."""
    cur_t = state_after.cur_turn
    p_owners = state_after.future_timeline[:, cur_t, 0]
    p_active = state_after.active_mask

    p0_alive = jnp.any((p_owners == 0) & p_active)
    p1_alive = jnp.any((p_owners == 1) & p_active)

    player_alive = jnp.where(player_id == 0, p0_alive, p1_alive)
    opponent_alive = jnp.where(player_id == 0, p1_alive, p0_alive)

    newly_done = state_after.done & ~state_before.done
    win_reward = newly_done & player_alive & ~opponent_alive
    loss_reward = newly_done & ~player_alive

    win_val = float(cfg.get("win_reward", 1.0))
    loss_val = float(cfg.get("loss_reward", -1.0))
    terminal_r = jnp.where(win_reward, win_val,
                           jnp.where(loss_reward, loss_val, 0.0))

    def potential(s: JaxEnvState) -> jnp.ndarray:
        t = s.cur_turn
        owners = s.future_timeline[:, t, 0]
        ships = s.future_timeline[:, t, 1]
        active = s.active_mask
        player_owned = (owners == player_id) & active
        opp_owned = (owners >= 0) & (owners != player_id) & active
        own_ships = jnp.sum(jnp.where(player_owned, ships, 0.0))
        opp_ships = jnp.sum(jnp.where(opp_owned, ships, 0.0))
        own_prod = jnp.sum(jnp.where(player_owned, s.planet_production, 0.0))
        opp_prod = jnp.sum(jnp.where(opp_owned, s.planet_production, 0.0))
        own_planets = jnp.sum(player_owned.astype(jnp.float32))
        opp_planets = jnp.sum(opp_owned.astype(jnp.float32))
        return (
            0.0010 * (own_ships - opp_ships)
            + 0.0100 * (own_prod - opp_prod)
            + 0.0200 * (own_planets - opp_planets)
        )

    shaping_coef = float(cfg.get("reward_shaping_coef", 0.05))
    shaped_delta = jnp.clip(potential(state_after) - potential(state_before), -0.05, 0.05)
    shaped_r = shaping_coef * shaped_delta
    return terminal_r + jnp.where(newly_done, 0.0, shaped_r)


class OrbitWarsPureJaxEnv:
    """A wrapper for Orbit Wars that makes reset and step fully jittable.
    
    It uses jax.pure_callback to call out to the host for python-based 
    reset (which relies on the Kaggle engine/generation) and precomputations.
    """
    def __init__(self, episode_steps: int = 500, ship_speed: float = 6.0):
        self.episode_steps = episode_steps
        self.ship_speed = ship_speed
        self.dummy_state = jax_orbit_wars_reset(0, as_jax=True)

    def reset(self, key: jax.Array) -> tuple[ObsBatch, JaxEnvState]:
        """Jittable reset using pure_callback."""
        seed = jax.random.randint(key, (), 0, 2**31 - 1)
        
        def _host_reset(s: int) -> JaxEnvState:
            return jax_orbit_wars_reset(int(s), as_jax=False)

        state = jax.pure_callback(
            _host_reset,
            self.dummy_state,
            seed,
            vmap_method="sequential",
        )
        obs = extract_obs_v9_jax(state, player_id=0)
        return obs, state

    def step(
        self, 
        key: jax.Array, 
        state: JaxEnvState, 
        actions_p0: jnp.ndarray, 
        owned_p0: jnp.ndarray,
        actions_p1: jnp.ndarray,
        owned_p1: jnp.ndarray,
    ) -> tuple[ObsBatch, JaxEnvState, jnp.ndarray, jnp.ndarray, dict]:
        """Jittable step without callbacks."""
        
        inert_actions = jnp.full(actions_p0.shape, -1, dtype=jnp.int32)
        inert_owned = jnp.full(owned_p0.shape, -1, dtype=jnp.int32)
        
        # 4-player format: shape (4, S) where S is the action size
        actions_4p = jnp.stack([actions_p0, actions_p1, inert_actions, inert_actions], axis=0)
        owned_4p = jnp.stack([owned_p0, owned_p1, inert_owned, inert_owned], axis=0)

        # 1. Pure JAX step
        next_state = jax_orbit_wars_step(
            state,
            actions_4p,
            owned_4p,
        )
        
        # 2. Rewards
        reward = compute_reward(state, next_state, player_id=jnp.array(0, dtype=jnp.int32), cfg={
            "win_reward": 1.0,
            "loss_reward": -1.0,
            "reward_shaping_coef": 0.05,
        })
        
        # Done flag
        done = next_state.done
        
        next_obs = extract_obs_v9_jax(next_state, player_id=0)
        
        # 3. Auto-reset
        def reset_fn():
            return self.reset(key)
            
        def keep_fn():
            return next_obs, next_state
            
        obs_out, state_out = jax.lax.cond(done, reset_fn, keep_fn)
        
        return obs_out, state_out, reward, done, {}


In [ ]:
%%writefile src/__init__.py
# Make src a package.


In [ ]:
%%writefile train.py
import argparse
import jax
import yaml
import time
import os
import equinox as eqx
from pathlib import Path
from src.ppo import make_train


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", type=str, default="default_cfg.yaml")
    return parser.parse_args()


def load_config(path: str):
    with open(path, "r") as f:
        data = yaml.safe_load(f)
    
    ppo_data = data.get("ppo", {})
    env_data = data.get("env", {})
    model_data = data.get("model", {})
    
    num_envs = int(ppo_data.get("num_envs", env_data.get("num_envs", 4)))
    rollout_steps = int(ppo_data.get("rollout_steps", env_data.get("rollout_steps", 128)))
    total_updates = int(ppo_data.get("total_updates", 500))
    
    flat_config = {
        "LR": float(ppo_data.get("lr", ppo_data.get("pi_lr", 2.5e-4))),
        "NUM_ENVS": num_envs,
        "NUM_STEPS": rollout_steps,
        "TOTAL_TIMESTEPS": int(total_updates * rollout_steps * num_envs),
        "UPDATE_EPOCHS": int(ppo_data.get("epochs", ppo_data.get("train_pi_iters", 4))),
        "NUM_MINIBATCHES": int(ppo_data.get("num_minibatches", 4)),
        "GAMMA": float(ppo_data.get("gamma", 0.99)),
        "GAE_LAMBDA": float(ppo_data.get("gae_lambda", 0.95)),
        "CLIP_EPS": float(ppo_data.get("clip_coef", 0.2)),
        "ENT_COEF": float(ppo_data.get("ent_coef", 0.01)),
        "VF_COEF": float(ppo_data.get("vf_coef", 0.5)),
        "MAX_GRAD_NORM": float(ppo_data.get("max_grad_norm", 0.5)),
        "ANNEAL_LR": True,
        
        # Env specific
        "EPISODE_STEPS": int(env_data.get("episode_steps", 500)),
        "SHIP_SPEED": float(env_data.get("ship_speed", 6.0)),
        
        # Model specific
        "D_MODEL": int(model_data.get("hidden_size", model_data.get("d_model", 64))),
        "NUM_HEADS": int(model_data.get("num_heads", 4)),
        "NUM_LAYERS": int(model_data.get("num_layers", 5)),
    }
    return flat_config


def main():
    args = parse_args()
    config = load_config(args.config)
    
    print(f"Starting JAX PPO training with config: {config}")
    
    rng = jax.random.PRNGKey(42)
    init_fn, update_fn = make_train(config)
    
    print("Compiling network and initial environment state...")
    runner_state = init_fn(rng)
    
    print("JIT Compiling update step...")
    update_fn_jit = jax.jit(update_fn)
    
    # Warmup compilation run
    runner_state, metrics = update_fn_jit(runner_state)
    jax.block_until_ready(metrics)
    print("Compilation finished. Starting training loop.")
    
    num_updates = config["TOTAL_TIMESTEPS"] // config["NUM_STEPS"] // config["NUM_ENVS"]
    steps_per_update = config["NUM_ENVS"] * config["NUM_STEPS"]
    
    os.makedirs("checkpoints", exist_ok=True)
    start_time = time.time()
    
    for update_idx in range(1, num_updates + 1):
        runner_state, metrics = update_fn_jit(runner_state)
        jax.block_until_ready(metrics)
        
        if update_idx % 5 == 0 or update_idx == 1:
            elapsed = time.time() - start_time
            sps = (update_idx * steps_per_update) / elapsed
            print(f"Update {update_idx:04d}/{num_updates} | SPS: {sps:.1f}")
            print(f"  Reward:     {metrics['reward']:.3f}")
            print(f"  Loss:       {metrics['loss']:.3f} (P: {metrics['policy_loss']:.3f}, V: {metrics['value_loss']:.3f}, E: {metrics['entropy']:.3f})")
            print(f"  Approx KL:  {metrics['approx_kl']:.4f}")
            print(f"  Clip Frac:  {metrics['clip_frac']:.4f}")
            print(f"  Expl. Var:  {metrics['explained_variance']:.4f}")
            print("-" * 50)
            
        if update_idx % 100 == 0:
            train_state = runner_state[0]
            ckpt_path = f"checkpoints/ckpt_{update_idx:04d}.eqx"
            eqx.tree_serialise_leaves(ckpt_path, train_state.model)
            print(f"Saved checkpoint to {ckpt_path}")

    # Save final model
    train_state = runner_state[0]
    final_path = "checkpoints/model_final.eqx"
    eqx.tree_serialise_leaves(final_path, train_state.model)
    print(f"Training finished! Final model saved to {final_path}")


if __name__ == "__main__":
    main()


## Configuration


In [ ]:
%%writefile kaggle_cfg.yaml

env:
  num_envs: 4
  rollout_steps: 128
  episode_steps: 500
  ship_speed: 6.0
model:
  d_model: 96
  num_heads: 4
  num_layers: 3
  bucket_count: 4
ppo:
  total_updates: 100
  train_pi_iters: 1
  minibatch_size: 128
  gamma: 0.99
  gae_lambda: 0.95
  clip_coef: 0.2
  ent_coef: 0.01
  vf_coef: 0.5
  pi_lr: 0.00025


## Train


In [ ]:
import os
# Optionally set JAX to use specific GPUs
# os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'
!python train.py --config kaggle_cfg.yaml
